# TNBC Model — Evaluation, Baselines, and Figures

This notebook loads the trained LOCO CV checkpoints from `tnbc.ipynb`, runs GNN inference per fold, trains Random Forest and ElasticNet baselines, computes DrEval-style residualised metrics, and generates all publication figures. Run `tnbc.ipynb` first to produce model checkpoints and `loco_cv_results_cell_line.pkl`.

In [ ]:
import json
import pickle
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from pathlib import Path
from scipy.stats import friedmanchisquare, pearsonr, spearmanr, wilcoxon
from sklearn.metrics import mean_squared_error, r2_score

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch_geometric.data import Batch
from torch_geometric.nn import TransformerConv, GINConv, global_mean_pool, global_max_pool
from torch_scatter import scatter_softmax
from tqdm import tqdm

warnings.filterwarnings('ignore')

current_dir = Path.cwd()
project_root = current_dir
for candidate in [current_dir, current_dir.parent, current_dir.parent.parent]:
    if (candidate / "data" / "raw").exists():
        project_root = candidate
        break

output_dir = project_root / "results"
output_dir.mkdir(parents=True, exist_ok=True)
viz_dir = output_dir / "viz"
viz_dir.mkdir(parents=True, exist_ok=True)

### Plot Style Configuration

In [ ]:
# Reference style: monochrome, clean, publication-ready
plt.rcParams.update({
    'figure.dpi': 300, 'savefig.dpi': 300, 'savefig.bbox': 'tight',
    'font.family': 'sans-serif', 'font.sans-serif': ['Helvetica', 'Arial', 'DejaVu Sans'],
    'font.size': 10, 'axes.labelsize': 11, 'axes.titlesize': 12,
    'axes.linewidth': 1.0, 'axes.spines.top': False, 'axes.spines.right': False,
    'xtick.labelsize': 9, 'ytick.labelsize': 9, 'legend.fontsize': 9,
    'lines.linewidth': 1.5, 'lines.markersize': 6,
    'grid.alpha': 0.3, 'grid.linestyle': '--', 'grid.linewidth': 0.5,
})

# Hatching patterns for grouped bars and box plots (reference style)
HATCH_STYLES = {
    'Phase 1': '',
    'Phase 2': '/',
    'Phase 3': 'xx',
    'Phase 1 GNN': '', 'Phase 3 GNN': '/', 'Random Forest': 'xx', 'ElasticNet': '..',
}
LINE_STYLES = {'Phase 1': '-', 'Phase 2': '--', 'Phase 3': '-.', 'Phase 3 (resid)': ':'}
MARKERS = {'Phase 1': 'o', 'Phase 2': 's', 'Phase 3': '^', 'Phase 3 (resid)': 'D'}

### Shared Helper Functions

In [ ]:
def _label_bars(ax, bars, vals, fmt='.2f', fontsize=6):
    for b, v in zip(bars, vals):
        if np.isfinite(v):
            h = b.get_height()
            off = 0.02 * (ax.get_ylim()[1] - ax.get_ylim()[0])
            y = h + off if h >= 0 else h - off
            va = 'bottom' if h >= 0 else 'top'
            ax.text(b.get_x() + b.get_width()/2, y, f'{v:{fmt}}', ha='center', va=va, fontsize=fontsize)

def _best_legend_loc(ax):
    """Pick legend corner with most empty space based on bar occupancy.
    Returns one of 'upper right', 'upper left', 'lower right', 'lower left'.
    """
    patches = [p for p in ax.patches if hasattr(p, 'get_height') and p.get_width() > 0 and abs(p.get_height()) > 1e-9]
    if not patches:
        return 'upper right'
    xmin, xmax = ax.get_xlim()
    ymin, ymax = ax.get_ylim()
    dx, dy = xmax - xmin or 1, ymax - ymin or 1
    x_mid = (xmin + xmax) / 2

    h, w = abs(patches[0].get_height()), abs(patches[0].get_width())
    is_horizontal = h < 0.5 * dy and w > 0.2 * dx

    if is_horizontal:
        lefts = [p.get_x() for p in patches]
        rights = [p.get_x() + p.get_width() for p in patches]
        tops = [p.get_y() + p.get_height() for p in patches]
        bottoms = [p.get_y() for p in patches]
        right_empty = xmax - max(rights)
        left_empty = min(lefts) - xmin
        top_empty = ymax - max(tops)
        bottom_empty = min(bottoms) - ymin
        # Use area (product) so we pick corner with most space in BOTH dimensions
        candidates = [
            ('upper right', right_empty * top_empty),
            ('upper left', left_empty * top_empty),
            ('lower right', right_empty * bottom_empty),
            ('lower left', left_empty * bottom_empty),
        ]
        return max(candidates, key=lambda t: t[1])[0]

    # Vertical bars: need both vertical and horizontal clearance per corner
    y_mid = (ymin + ymax) / 2
    centers_x = [p.get_x() + p.get_width() / 2 for p in patches]
    rights = [p.get_x() + p.get_width() for p in patches]
    tops = [p.get_y() + p.get_height() for p in patches]
    bottoms = [p.get_y() for p in patches]

    def _max_top(in_right_half):
        vals = [t for cx, t in zip(centers_x, tops) if (cx > x_mid) == in_right_half]
        return max(vals) if vals else ymin

    def _min_bottom(in_right_half):
        vals = [b for cx, b in zip(centers_x, bottoms) if (cx > x_mid) == in_right_half]
        return min(vals) if vals else ymax

    def _max_right(in_top_half):
        vals = [r for t, r in zip(tops, rights) if (t > y_mid) == in_top_half]
        return max(vals) if vals else xmin

    def _min_left(in_top_half):
        vals = [p.get_x() for p, t in zip(patches, tops) if (t > y_mid) == in_top_half]
        return min(vals) if vals else xmax

    ur_vert = ymax - _max_top(True)
    ur_horiz = xmax - _max_right(True)
    ul_vert = ymax - _max_top(False)
    ul_horiz = _min_left(True) - xmin
    lr_vert = _min_bottom(True) - ymin
    lr_horiz = xmax - _max_right(False)
    ll_vert = _min_bottom(False) - ymin
    ll_horiz = _min_left(False) - xmin

    candidates = [
        ('upper right', ur_vert * ur_horiz),
        ('upper left', ul_vert * ul_horiz),
        ('lower right', lr_vert * lr_horiz),
        ('lower left', ll_vert * ll_horiz),
    ]
    return max(candidates, key=lambda t: t[1])[0]

def _mean_per_fold_r2(df, y_true_col='y_true', y_pred_col='y_pred', min_per_fold=3):
    """Mean of per-fold R² (standard LOCO metric). Returns None if fold_id missing or insufficient folds."""
    if df is None or df.empty or 'fold_id' not in df.columns or y_true_col not in df.columns or y_pred_col not in df.columns:
        return None
    r2_per_fold = []
    for fid in sorted(df['fold_id'].unique()):
        g = df[df['fold_id'] == fid]
        if len(g) >= min_per_fold:
            r2_per_fold.append(r2_score(g[y_true_col], g[y_pred_col]))
    return np.nanmean(r2_per_fold) if r2_per_fold else None

def _mean_per_fold_resid_r2(df):
    """Mean per-fold residualized R². df must have y_true_resid, y_pred_resid, fold_id."""
    if df is None or 'y_true_resid' not in df.columns or 'y_pred_resid' not in df.columns:
        return None
    return _mean_per_fold_r2(df, 'y_true_resid', 'y_pred_resid')

def _mean_per_fold_pearson(df, y_true_col='y_true', y_pred_col='y_pred', min_per_fold=3):
    """Mean of per-fold Pearson r. Returns None if fold_id missing or insufficient folds."""
    if df is None or df.empty or 'fold_id' not in df.columns or y_true_col not in df.columns or y_pred_col not in df.columns:
        return None
    r_per_fold = []
    for fid in sorted(df['fold_id'].unique()):
        g = df[df['fold_id'] == fid]
        if len(g) >= min_per_fold:
            r, _ = pearsonr(g[y_true_col], g[y_pred_col])
            if np.isfinite(r):
                r_per_fold.append(r)
    return np.nanmean(r_per_fold) if r_per_fold else None

## GNN Predictions and DrEval Metrics

Loads Phase 1/2/3 checkpoints per LOCO fold, runs inference on each held-out TNBC cell line, and computes DrEval-style metrics (drug-mean residualised Pearson, R², per-drug Spearman). Saves predictions and metrics to `results/viz/`.

In [ ]:
# Generate predictions by loading models from .pt files
viz_dir = output_dir / "viz"
viz_dir.mkdir(parents=True, exist_ok=True)
gnn_pred_path = viz_dir / "tnbc_gnn_loco_predictions.csv"

if not gnn_pred_path.exists():
    if torch.cuda.is_available():
        device = torch.device('cuda')
    else:
        device = torch.device('cpu')

    with open(output_dir / "loco_cv_results_cell_line.pkl", 'rb') as f:
        loco_data = pickle.load(f)
    _fr = loco_data['fold_results']
    all_tnbc_cells = sorted(set(r['test_cell'] for r in _fr))

    models_dir_eval = output_dir / "models" / "tnbc"

    with open(project_root / "data" / "processed" / "drug_graphs.pkl", 'rb') as f:
        drug_graphs = pickle.load(f)

    data_dir = project_root / "data" / "raw"
    gdsc2_df = pd.read_excel(data_dir / "GDSC2 Fitted Dose Response Oct 27 2023.xlsx")
    model_df = pd.read_csv(data_dir / "DepMap Model Data.csv")
    pathway_scores_raw = pd.read_csv(data_dir / "cell_ge.csv", index_col=0)
    pathway_names = pathway_scores_raw.index.tolist()
    pathway_data = pathway_scores_raw.T.reset_index()
    pathway_data.columns = ['CellLineName'] + pathway_names
    rmse_col = [col for col in gdsc2_df.columns if 'RMSE' in col.upper()][0]
    gdsc2_filtered = gdsc2_df[gdsc2_df[rmse_col] < 0.3].copy()
    drug_response = gdsc2_filtered[['DRUG_NAME', 'CELL_LINE_NAME', 'LN_IC50', 'COSMIC_ID']].copy()
    drug_response.columns = ['DrugName', 'CellLineName', 'LN_IC50', 'COSMICID']
    cell_name_to_modelid = dict(zip(model_df['StrippedCellLineName'].str.upper().str.replace('-', '').str.replace('_', ''), model_df['ModelID']))
    cosmic_to_modelid = model_df.drop_duplicates(subset='COSMICID', keep='first').set_index('COSMICID')['ModelID'].to_dict()
    pathway_data['ModelID'] = pathway_data['CellLineName'].apply(lambda x: cell_name_to_modelid.get(str(x).upper().replace('-', '').replace('_', ''), None))
    pathway_data = pathway_data[pathway_data['ModelID'].notna()].copy()
    drug_response['ModelID'] = drug_response['COSMICID'].apply(lambda x: cosmic_to_modelid.get(x, None))
    drug_response.loc[drug_response['ModelID'].isna(), 'ModelID'] = drug_response.loc[drug_response['ModelID'].isna(), 'CellLineName'].apply(lambda x: cell_name_to_modelid.get(str(x).upper().replace('-', '').replace('_', ''), None))
    pan_cancer_pathway = pathway_data.merge(drug_response[['DrugName', 'ModelID', 'LN_IC50']], on='ModelID', how='inner')
    pan_cancer_pathway = pan_cancer_pathway.rename(columns={'DrugName': 'DRUG_NAME'})
    valid_drugs = set(drug_graphs.keys())
    pan_cancer_pathway = pan_cancer_pathway[pan_cancer_pathway['DRUG_NAME'].isin(valid_drugs)].copy()
    model_cols = ['ModelID', 'StrippedCellLineName', 'OncotreeLineage', 'OncotreePrimaryDisease', 'OncotreeSubtype']
    for col in ['PatientSubtypeFeatures', 'ModelSubtypeFeatures']:
        if col in model_df.columns:
            model_cols.append(col)
    model_cols = [c for c in model_cols if c in model_df.columns]
    pan_cancer_pathway = pan_cancer_pathway.merge(model_df[model_cols], on='ModelID', how='left')
    mutation_file = data_dir / "Omics Somatic Mutations.csv"
    if not mutation_file.exists():
        mutation_file = data_dir / "OmicsSomaticMutations.csv"
    mutations_df = pd.read_csv(mutation_file, low_memory=False)
    priority_genes = ['BRCA1', 'BRCA2', 'TP53', 'PIK3CA', 'PTEN', 'RB1', 'EGFR', 'MYC', 'CDKN2A', 'GATA3', 'MAP3K1', 'KRAS', 'ARID1A', 'NF1', 'ERBB2', 'ESR1', 'PGR', 'AKT1']
    gene_col = 'HugoSymbol' if 'HugoSymbol' in mutations_df.columns else 'Hugo_Symbol'
    if 'LikelyLoF' in mutations_df.columns:
        mutations_df['is_mutated'] = mutations_df['LikelyLoF'].fillna(False).astype(int)
    elif 'TranscriptLikelyLof' in mutations_df.columns:
        mutations_df['is_mutated'] = mutations_df['TranscriptLikelyLof'].fillna(False).astype(int)
    elif 'VepImpact' in mutations_df.columns:
        mutations_df['is_mutated'] = mutations_df['VepImpact'].isin(['HIGH', 'MODERATE']).astype(int)
    else:
        mutations_df['is_mutated'] = 1
    mutation_matrix = mutations_df[mutations_df[gene_col].isin(priority_genes)].pivot_table(index='ModelID', columns=gene_col, values='is_mutated', aggfunc='max', fill_value=0)
    mutation_matrix = mutation_matrix.reindex(columns=priority_genes, fill_value=0)
    mutation_matrix['total_mutation_burden'] = mutation_matrix.sum(axis=1)
    mutation_features = mutation_matrix.reset_index()
    mutation_cols = [c for c in mutation_features.columns if c != 'ModelID']
    pan_cancer_pathway = pan_cancer_pathway.merge(mutation_features[['ModelID'] + mutation_cols], on='ModelID', how='left')
    pan_cancer_pathway[mutation_cols] = pan_cancer_pathway[mutation_cols].fillna(0)
    proteomics_file = data_dir / "Breast Cancer Proteomic Dynamics (2).csv"
    if proteomics_file.exists():
        try:
            proteomics_raw = pd.read_csv(proteomics_file, sep=';', on_bad_lines='skip', low_memory=False)
        except Exception:
            proteomics_raw = pd.read_csv(proteomics_file, sep=';', on_bad_lines='skip', low_memory=False, encoding='latin-1')
        protein_ids = proteomics_raw.iloc[:, 0].astype(str) + ';' + proteomics_raw.iloc[:, 1].astype(str)
        proteomics_raw = proteomics_raw.set_index(protein_ids)
        proteomics_raw = proteomics_raw.iloc[:, 2:]
        def _to_float(val):
            if pd.isna(val) or val == '': return np.nan
            s = str(val).strip().replace(',', '.')
            try: return float(s)
            except ValueError: return np.nan
        for c in proteomics_raw.columns:
            proteomics_raw[c] = proteomics_raw[c].apply(_to_float)
        proteomics_T = proteomics_raw.T
        def cell_col_to_base(name):
            base = str(name).split('_')[0] if '_' in str(name) else str(name)
            return base.upper().replace('-', '').replace(' ', '')
        cell_to_base = {c: cell_col_to_base(c) for c in proteomics_T.index}
        proteomics_T.index = proteomics_T.index.map(lambda x: cell_name_to_modelid.get(cell_to_base[x], None))
        proteomics_T = proteomics_T[proteomics_T.index.notna()]
        proteomics_T.index = proteomics_T.index.astype(str)
        proteomics_by_modelid = proteomics_T.groupby(proteomics_T.index).median()
        proteomics_by_modelid = proteomics_by_modelid.loc[:, ~proteomics_by_modelid.columns.duplicated()]
        frac_observed = proteomics_by_modelid.notna().mean(axis=0)
        protein_cols = proteomics_by_modelid.columns[frac_observed >= 0.70].tolist()
        protein_cols = list(dict.fromkeys(protein_cols))
        proteomics_by_modelid = proteomics_by_modelid[protein_cols]
        proteomics_by_modelid_full = proteomics_by_modelid.copy()
        protein_cols_full = protein_cols.copy()
    else:
        proteomics_by_modelid_full = pd.DataFrame()
        protein_cols_full = []
    cfg = {'phase2_lineage': 'Breast', 'phase3_subtype_col': 'ModelSubtypeFeatures', 'phase3_subtype_kw': 'TNBC'}
    breast_pathway = pan_cancer_pathway[pan_cancer_pathway['OncotreeLineage'] == cfg['phase2_lineage']].copy()
    _sub_col = cfg['phase3_subtype_col']
    _sub_kw = cfg['phase3_subtype_kw']
    if _sub_col in breast_pathway.columns:
        subtype_mask = breast_pathway[_sub_col].astype(str).str.contains(_sub_kw, case=False, na=False)
    else:
        psf = breast_pathway['PatientSubtypeFeatures'].astype(str).str.contains(_sub_kw, case=False, na=False) if 'PatientSubtypeFeatures' in breast_pathway.columns else pd.Series(False, index=breast_pathway.index)
        msf = breast_pathway['ModelSubtypeFeatures'].astype(str).str.contains(_sub_kw, case=False, na=False) if 'ModelSubtypeFeatures' in breast_pathway.columns else pd.Series(False, index=breast_pathway.index)
        subtype_mask = psf | msf
    tnbc_pathway = breast_pathway[subtype_mask].copy()
    drug_name_col = 'DRUG_NAME'
    pathway_cols = [c for c in pathway_names if c in tnbc_pathway.columns]
    mutation_cols = [c for c in mutation_cols if c in tnbc_pathway.columns]
    p1_ckpt_path = models_dir_eval / "loco_shared" / "phase1.pt"
    if p1_ckpt_path.exists():
        ckpt = torch.load(p1_ckpt_path, map_location=device, weights_only=False)
        state = ckpt.get('model_state_dict', ckpt)
        total_cell_input_dim = state['reconstruction_head.2.weight'].shape[0]
        shared_n_proteins = state['protein_proj.weight'].shape[1] if 'protein_proj.weight' in state else 0
        if len(pathway_cols) + len(mutation_cols) != total_cell_input_dim:
            pathway_cols = [c for c in pathway_names if c in tnbc_pathway.columns][:total_cell_input_dim - len(mutation_cols)]
    else:
        total_cell_input_dim = len(pathway_cols) + len(mutation_cols)
        shared_n_proteins = 0
    shared_protein_cols = []

    class DrugEncoder(nn.Module):
        def __init__(self, node_feature_dim=9, edge_feature_dim=4, hidden_dim=256, dropout=0.3):
            super(DrugEncoder, self).__init__()
            if torch.cuda.is_available():
                self.device = torch.device('cuda')
            else:
                self.device = torch.device('cpu')
            gin_nn1 = nn.Sequential(nn.Linear(node_feature_dim, 128), nn.ReLU(), nn.LayerNorm(128), nn.Dropout(dropout), nn.Linear(128, 128))
            self.gin_conv1 = GINConv(gin_nn1, train_eps=True)
            self.gin_bn1 = nn.BatchNorm1d(128)
            gin_nn2 = nn.Sequential(nn.Linear(128, 128), nn.ReLU(), nn.LayerNorm(128), nn.Dropout(dropout), nn.Linear(128, 128))
            self.gin_conv2 = GINConv(gin_nn2, train_eps=True)
            self.gin_bn2 = nn.BatchNorm1d(128)
            self.transformer_conv1 = TransformerConv(node_feature_dim, 128, heads=4, dropout=dropout, edge_dim=edge_feature_dim)
            self.transformer_bn1 = nn.BatchNorm1d(128 * 4)
            self.transformer_conv2 = TransformerConv(128 * 4, 128, heads=4, concat=False, dropout=dropout, edge_dim=edge_feature_dim)
            self.transformer_bn2 = nn.BatchNorm1d(128)
            self.alpha = nn.Parameter(torch.tensor(0.5))
            self.proj = nn.Linear(128, hidden_dim)
            self.proj_bn = nn.BatchNorm1d(hidden_dim)
            self.dropout = nn.Dropout(dropout)
            self.relu = nn.ReLU()
            self.attention_weights = nn.Linear(hidden_dim, 1)
            self.to(self.device)
        def forward(self, data):
            x, edge_index, edge_attr, batch = data.x, data.edge_index, data.edge_attr, data.batch
            x = x.to(self.device, non_blocking=True)
            edge_index = edge_index.to(self.device, non_blocking=True)
            edge_attr = edge_attr.to(self.device, non_blocking=True)
            batch = batch.to(self.device, non_blocking=True)
            x_gin = self.gin_conv1(x, edge_index)
            x_gin = self.gin_bn1(x_gin)
            x_gin = self.relu(x_gin)
            x_gin = self.dropout(x_gin)
            x_gin = self.gin_conv2(x_gin, edge_index)
            x_gin = self.gin_bn2(x_gin)
            x_gin = self.relu(x_gin)
            x_transformer = self.transformer_conv1(x, edge_index, edge_attr)
            x_transformer = self.transformer_bn1(x_transformer)
            x_transformer = self.relu(x_transformer)
            x_transformer = self.dropout(x_transformer)
            x_transformer = self.transformer_conv2(x_transformer, edge_index, edge_attr)
            x_transformer = self.transformer_bn2(x_transformer)
            x_transformer = self.relu(x_transformer)
            alpha_sigmoid = torch.sigmoid(self.alpha)
            x_merged = alpha_sigmoid * x_gin + (1 - alpha_sigmoid) * x_transformer
            x = self.proj(x_merged)
            x = self.proj_bn(x)
            x = self.relu(x)
            attention_scores = self.attention_weights(x)
            attention_scores = scatter_softmax(attention_scores.squeeze(-1), batch, dim=0)
            attention_scores = attention_scores.unsqueeze(-1)
            x_weighted = x * attention_scores
            x_mean = global_mean_pool(x_weighted, batch)
            x_max = global_max_pool(x, batch)
            embedding = (x_mean + x_max) / 2
            return embedding

    class ExpandedCellEncoder(nn.Module):
        def __init__(self, input_dim=1329, output_dim=256):
            super().__init__()
            self.fc1 = nn.Linear(input_dim, 512)
            self.bn1 = nn.BatchNorm1d(512)
            self.dropout1 = nn.Dropout(0.4)
            self.fc2 = nn.Linear(512, output_dim)
            self.bn2 = nn.BatchNorm1d(output_dim)
            self.dropout2 = nn.Dropout(0.3)
            self.residual2 = nn.Linear(512, output_dim)
            self.skip = nn.Linear(input_dim, output_dim)
            self.relu = nn.ReLU()
            if torch.cuda.is_available():
                self.device = torch.device('cuda')
            else:
                self.device = torch.device('cpu')
            self.to(self.device)
        def forward(self, x):
            x = x.to(self.device)
            out1 = self.fc1(x)
            out1 = self.bn1(out1)
            out1 = self.relu(out1)
            out1 = self.dropout1(out1)
            out2 = self.fc2(out1)
            out2 = self.bn2(out2)
            res2 = self.residual2(out1)
            out2 = self.relu(out2 + res2)
            out2 = self.dropout2(out2)
            skip = self.skip(x)
            output = out2 + skip
            return output

    class DrugResponsePathwayGNN(nn.Module):
        def __init__(self, drug_node_dim=9, drug_edge_dim=4, cell_input_dim=1329, n_proteins=0, hidden_dim=256, dropout=0.3):
            super(DrugResponsePathwayGNN, self).__init__()
            if torch.cuda.is_available():
                self.device = torch.device('cuda')
            else:
                self.device = torch.device('cpu')
            self.n_proteins = n_proteins
            self.hidden_dim = hidden_dim
            self.cell_input_dim = cell_input_dim
            if n_proteins > 0:
                self.protein_proj = nn.Linear(n_proteins, 32)
                self.protein_dim = 32
            else:
                self.protein_proj = None
                self.protein_dim = 0
            self.drug_encoder = DrugEncoder(node_feature_dim=drug_node_dim, edge_feature_dim=drug_edge_dim, hidden_dim=hidden_dim, dropout=dropout)
            self.cell_encoder = ExpandedCellEncoder(input_dim=cell_input_dim, output_dim=hidden_dim)
            self.attention_query = nn.Linear(hidden_dim, hidden_dim)
            self.attention_key = nn.Linear(hidden_dim, hidden_dim)
            self.attention_value = nn.Linear(hidden_dim, hidden_dim)
            combined_dim = hidden_dim * 2 + self.protein_dim
            self.ic50_head = nn.Sequential(nn.Linear(combined_dim, 256), nn.ReLU(), nn.Dropout(0.3), nn.Linear(256, 128), nn.ReLU(), nn.Dropout(0.2), nn.Linear(128, 1))
            self.classification_head = nn.Sequential(nn.Linear(combined_dim, 256), nn.ReLU(), nn.Dropout(0.3), nn.Linear(256, 1))
            self.reconstruction_head = nn.Sequential(nn.Linear(combined_dim, 256), nn.ReLU(), nn.Linear(256, cell_input_dim))
            self.to(self.device)
        def integrate_with_attention(self, drug_emb, cell_emb):
            query = self.attention_query(drug_emb)
            key = self.attention_key(cell_emb)
            value = self.attention_value(cell_emb)
            attention_scores = torch.matmul(query.unsqueeze(1), key.unsqueeze(2))
            attention_scores = attention_scores / (self.hidden_dim ** 0.5)
            attention_weights = torch.softmax(attention_scores, dim=-1)
            attended_cell = attention_weights.squeeze(-1) * value
            return torch.cat([drug_emb, attended_cell], dim=1)
        def forward(self, drug_batch, cell_batch, proteins=None, has_proteomics=None, return_embeddings=False):
            drug_emb = self.drug_encoder(drug_batch)
            cell_emb = self.cell_encoder(cell_batch.to(self.device))
            combined = self.integrate_with_attention(drug_emb, cell_emb)
            if self.protein_proj is not None:
                if proteins is not None and proteins.numel() > 0:
                    prot_emb = self.protein_proj(proteins.to(self.device))
                    if has_proteomics is not None:
                        mask = has_proteomics.to(self.device)
                        prot_emb = prot_emb * mask
                else:
                    prot_emb = torch.zeros(cell_emb.size(0), self.protein_dim, device=self.device)
                combined = torch.cat([combined, prot_emb], dim=1)
            ic50_pred = self.ic50_head(combined)
            class_pred = self.classification_head(combined)
            recon_pred = self.reconstruction_head(combined)
            results = {'ic50': ic50_pred, 'classification': class_pred, 'reconstruction': recon_pred}
            if return_embeddings:
                results['embeddings'] = {'drug': drug_emb, 'cell': cell_emb, 'combined': combined}
            return results

    class DrugResponsePathwayDataset(Dataset):
        def __init__(self, dataframe, drug_graphs_dict, drug_col='DRUG_NAME', pathway_cols=None, mutation_cols=None, proteomics_by_modelid=None, protein_cols=None, protein_mean=None, protein_std=None):
            self.original_data = dataframe.copy()
            self.data = dataframe.reset_index(drop=True)
            self.drug_graphs = drug_graphs_dict
            self.drug_col = drug_col
            self.proteomics_by_modelid = proteomics_by_modelid
            self.protein_cols = protein_cols if protein_cols is not None else []
            self.protein_mean = np.array(protein_mean, dtype=np.float32) if protein_mean is not None else None
            self.protein_std = np.array(protein_std, dtype=np.float32) if protein_std is not None else None
            self.n_proteins = len(self.protein_cols)
            if pathway_cols is not None:
                self.pathway_cols = [c for c in pathway_cols if c in dataframe.columns]
            else:
                exclude_cols = ['ModelID', 'COSMICID', 'StrippedCellLineName', 'OncotreeLineage', 'OncotreePrimaryDisease', drug_col, 'SMILES', 'LN_IC50', 'CellLineName']
                numeric_cols = dataframe.select_dtypes(include=[np.number]).columns.tolist()
                self.pathway_cols = [c for c in numeric_cols if c not in exclude_cols]
            for col in self.pathway_cols:
                if not pd.api.types.is_numeric_dtype(dataframe[col]):
                    raise ValueError(f"Pathway column '{col}' is not numeric")
            if mutation_cols is not None:
                self.mutation_cols = [c for c in mutation_cols if c in dataframe.columns]
            else:
                self.mutation_cols = []
            self.all_feature_cols = self.pathway_cols + self.mutation_cols
        def __len__(self):
            return len(self.data)
        def __getitem__(self, idx):
            row = self.data.iloc[idx]
            drug_name = row[self.drug_col]
            if drug_name not in self.drug_graphs:
                raise ValueError(f"Drug {drug_name} not found in drug_graphs")
            drug_graph = self.drug_graphs[drug_name]['graph_data']
            pathway_scores = row[self.pathway_cols].values.astype(np.float32)
            mean = pathway_scores.mean()
            std = pathway_scores.std()
            if std > 1e-8:
                pathway_scores = (pathway_scores - mean) / std
            else:
                pathway_scores = np.zeros_like(pathway_scores)
            if len(self.mutation_cols) > 0:
                mutation_features = row[self.mutation_cols].values.astype(np.float32)
                cell_features = np.concatenate([pathway_scores, mutation_features])
            else:
                cell_features = pathway_scores
            if self.n_proteins > 0:
                model_id = str(row['ModelID'])
                if self.proteomics_by_modelid is not None and model_id in self.proteomics_by_modelid.index:
                    cell_row = self.proteomics_by_modelid.loc[model_id]
                    if isinstance(cell_row, pd.DataFrame):
                        cell_row = cell_row.median(axis=0)
                    vals = np.asarray(cell_row.values, dtype=np.float32)
                    if len(vals) > self.n_proteins:
                        vals = vals[:self.n_proteins]
                    elif len(vals) < self.n_proteins:
                        vals = np.pad(vals, (0, self.n_proteins - len(vals)), constant_values=0)
                    prots = np.nan_to_num(vals, nan=0.0, posinf=0.0, neginf=0.0)
                    if self.protein_mean is not None and self.protein_std is not None:
                        n = min(len(prots), len(self.protein_mean), len(self.protein_std))
                        prots = prots[:n].astype(np.float32)
                        mean_use = np.asarray(self.protein_mean[:n], dtype=np.float32)
                        std_safe = np.where(np.asarray(self.protein_std[:n]) > 1e-8, self.protein_std[:n], 1.0).astype(np.float32)
                        prots = (prots - mean_use) / std_safe
                        if n < self.n_proteins:
                            prots = np.pad(prots, (0, self.n_proteins - n), constant_values=0)
                    has_proteomics = 1.0
                else:
                    prots = np.zeros(self.n_proteins, dtype=np.float32)
                    has_proteomics = 0.0
                proteins_tensor = torch.tensor(prots, dtype=torch.float32)
                has_proteomics_tensor = torch.tensor([has_proteomics], dtype=torch.float32)
            else:
                proteins_tensor = torch.tensor([], dtype=torch.float32)
                has_proteomics_tensor = torch.tensor([0.0], dtype=torch.float32)
            cell_expr_tensor = torch.tensor(cell_features, dtype=torch.float32)
            ic50 = torch.tensor([row['LN_IC50']], dtype=torch.float32)
            return {'drug_graph': drug_graph, 'cell_expr': cell_expr_tensor, 'proteins': proteins_tensor, 'has_proteomics': has_proteomics_tensor, 'ic50': ic50, 'drug_name': drug_name, 'cell_id': row['ModelID']}

    def collate_fn(batch):
        drug_graphs = [item['drug_graph'] for item in batch]
        cell_exprs = torch.stack([item['cell_expr'] for item in batch])
        ic50s = torch.stack([item['ic50'] for item in batch])
        drug_names = [item['drug_name'] for item in batch]
        cell_ids = [item['cell_id'] for item in batch]
        drug_batch = Batch.from_data_list(drug_graphs)
        out = {'drug_batch': drug_batch, 'cell_batch': cell_exprs, 'ic50': ic50s, 'drug_names': drug_names, 'cell_ids': cell_ids}
        if 'proteins' in batch[0] and batch[0]['proteins'].numel() > 0:
            out['proteins'] = torch.stack([item['proteins'] for item in batch])
            out['has_proteomics'] = torch.stack([item['has_proteomics'] for item in batch])
        return out

    def select_proteins_by_ic50_corr(trainval_cells, proteomics_df, protein_cols, pathway_df, top_n=50, min_cells=8):
        cell_mean_ic50 = pathway_df[pathway_df['ModelID'].isin(trainval_cells)].groupby('ModelID')['LN_IC50'].mean()
        cells_with_both = [c for c in trainval_cells if c in proteomics_df.index and c in cell_mean_ic50.index]
        cells_with_both = list(dict.fromkeys(cells_with_both))
        if len(cells_with_both) < min_cells:
            prot_sub = proteomics_df.loc[cells_with_both, protein_cols] if cells_with_both else proteomics_df[protein_cols]
            return prot_sub.var(axis=0).sort_values(ascending=False).head(top_n).index.tolist()
        correlations = {}
        for prot in protein_cols:
            vals = proteomics_df.loc[cells_with_both, prot]
            if isinstance(vals, pd.DataFrame):
                vals = vals.iloc[:, 0]
            ic50s = cell_mean_ic50.loc[cells_with_both]
            valid = vals.notna()
            n_valid = int(valid.sum())
            if n_valid >= min_cells:
                r, _ = pearsonr(vals[valid].values, ic50s[valid].values)
                correlations[prot] = abs(r)
        if not correlations:
            return protein_cols[:top_n]
        return pd.Series(correlations).sort_values(ascending=False).head(top_n).index.tolist()

    def build_prot_kw(fold_protein_cols, fold_proteomics, trainval_cells):
        if not fold_protein_cols:
            return {}
        trainval_with_prot = [c for c in trainval_cells if c in fold_proteomics.index]
        if trainval_with_prot:
            pt = fold_proteomics.loc[trainval_with_prot]
            pm = pt.mean(axis=0).values.astype(np.float32)
            ps = np.where(pt.std(axis=0).values > 1e-8, pt.std(axis=0).values, 1.0).astype(np.float32)
        else:
            pm = np.zeros(len(fold_protein_cols), dtype=np.float32)
            ps = np.ones(len(fold_protein_cols), dtype=np.float32)
        return dict(proteomics_by_modelid=fold_proteomics, protein_cols=fold_protein_cols, protein_mean=pm, protein_std=ps)

    _shared_protein_cols = select_proteins_by_ic50_corr(trainval_cells=all_tnbc_cells, proteomics_df=proteomics_by_modelid_full, protein_cols=protein_cols_full, pathway_df=tnbc_pathway, top_n=50, min_cells=8) if len(protein_cols_full) > 0 else []
    shared_protein_cols = _shared_protein_cols[:shared_n_proteins] if shared_n_proteins > 0 else []
    if shared_n_proteins > 0 and len(shared_protein_cols) < shared_n_proteins:
        raise RuntimeError(f"Checkpoint expects {shared_n_proteins} proteins but select_proteins returned {len(shared_protein_cols)}. Ensure proteomics data matches tnbc.")

    loco_shared_dir = models_dir_eval / "loco_shared"
    p1_ckpt = loco_shared_dir / "phase1.pt"
    p2_ckpt = loco_shared_dir / "phase2.pt"
    shared_prot_kw = build_prot_kw(shared_protein_cols, proteomics_by_modelid_full[shared_protein_cols], all_tnbc_cells) if shared_n_proteins > 0 else {}
    model_p1 = DrugResponsePathwayGNN(cell_input_dim=total_cell_input_dim, n_proteins=shared_n_proteins).to(device)
    model_p1.load_state_dict(torch.load(p1_ckpt, map_location=device, weights_only=False)['model_state_dict'])
    model_p1.eval()
    has_p2 = p2_ckpt.exists()
    if has_p2:
        model_p2 = DrugResponsePathwayGNN(cell_input_dim=total_cell_input_dim, n_proteins=shared_n_proteins).to(device)
        model_p2.load_state_dict(torch.load(p2_ckpt, map_location=device, weights_only=False)['model_state_dict'])
        model_p2.eval()

    rows = []
    for r in tqdm(_fr, desc="Collecting predictions"):
        fold_id = r['fold']
        test_cell = r['test_cell']
        trainval_cells = set(all_tnbc_cells) - {test_cell}
        fold_protein_cols = select_proteins_by_ic50_corr(trainval_cells=list(trainval_cells), proteomics_df=proteomics_by_modelid_full, protein_cols=protein_cols_full, pathway_df=tnbc_pathway, top_n=50, min_cells=8) if len(protein_cols_full) > 0 else []
        fold_prot_kw = build_prot_kw(fold_protein_cols, proteomics_by_modelid_full[fold_protein_cols], list(trainval_cells)) if len(fold_protein_cols) > 0 else {}
        test_idx = tnbc_pathway[tnbc_pathway['ModelID'] == test_cell].index.values
        test_data = tnbc_pathway.loc[test_idx].reset_index(drop=True)
        test_dataset_p1 = DrugResponsePathwayDataset(test_data, drug_graphs, drug_col=drug_name_col, pathway_cols=pathway_cols, mutation_cols=mutation_cols, **shared_prot_kw)
        test_dataset_p3 = DrugResponsePathwayDataset(test_data, drug_graphs, drug_col=drug_name_col, pathway_cols=pathway_cols, mutation_cols=mutation_cols, **fold_prot_kw)
        test_loader_p1 = DataLoader(test_dataset_p1, batch_size=64, shuffle=False, collate_fn=collate_fn, num_workers=0)
        test_loader_p3 = DataLoader(test_dataset_p3, batch_size=64, shuffle=False, collate_fn=collate_fn, num_workers=0)

        model_p3 = DrugResponsePathwayGNN(cell_input_dim=total_cell_input_dim, n_proteins=len(fold_protein_cols)).to(device)
        ckpt = torch.load(models_dir_eval / f"loco_fold{fold_id}" / "phase3.pt", map_location=device, weights_only=False)
        model_p3.load_state_dict(ckpt['model_state_dict'])
        model_p3.eval()

        def _run_model(loader, model, return_classification=False):
            preds_list, ids_list, y_list = [], [], []
            class_list = [] if return_classification else None
            with torch.no_grad():
                for batch in loader:
                    drug_batch = batch['drug_batch'].to(device)
                    cell_batch = batch['cell_batch'].to(device)
                    proteins = batch.get('proteins')
                    has_proteomics = batch.get('has_proteomics')
                    outputs = model(drug_batch, cell_batch, proteins=proteins, has_proteomics=has_proteomics)
                    preds = outputs['ic50'].squeeze()
                    if preds.dim() == 0:
                        preds = preds.unsqueeze(0)
                    preds_list.append(preds.cpu().numpy().flatten())
                    y_list.append(batch['ic50'].cpu().numpy().flatten())
                    ids_list.extend(zip(batch['cell_ids'], batch['drug_names']))
                    if return_classification:
                        cl = outputs['classification'].squeeze().cpu().numpy().flatten()
                        if np.ndim(cl) == 0:
                            cl = np.array([cl])
                        class_list.append(cl)
            out = (np.concatenate(preds_list), np.concatenate(y_list), ids_list)
            if return_classification:
                out = out + (np.concatenate(class_list),)
            return out

        preds_p1, y_true_arr, ids = _run_model(test_loader_p1, model_p1)
        preds_p3, _, _, class_logits_p3 = _run_model(test_loader_p3, model_p3, return_classification=True)
        preds_p2 = _run_model(test_loader_p1, model_p2)[0] if has_p2 else None
        trainval_df = tnbc_pathway[tnbc_pathway['ModelID'].isin(trainval_cells)]
        drug_medians = trainval_df.groupby(drug_name_col)['LN_IC50'].median().to_dict()
        global_median = np.median(tnbc_pathway['LN_IC50'])
        class_probs = 1.0 / (1.0 + np.exp(-class_logits_p3))
        y_pred_class = (class_probs >= 0.5).astype(int)
        for i, (cn, dn) in enumerate(ids):
            if i < len(preds_p3):
                thresh = drug_medians.get(dn, global_median)
                y_true_class = 1 if y_true_arr[i] < thresh else 0
                row = {'cell_line': cn, 'drug_name': dn, 'y_true': float(y_true_arr[i]), 'y_pred': float(preds_p3[i]), 'y_pred_p1': float(preds_p1[i]), 'fold_id': fold_id, 'y_true_class': y_true_class, 'y_pred_class': int(y_pred_class[i])}
                if has_p2:
                    row['y_pred_p2'] = float(preds_p2[i])
                rows.append(row)

    df_loco = pd.DataFrame(rows)

    df_loco['y_mean'] = np.nan
    for fold_id in df_loco['fold_id'].unique():
        test_cell_f = df_loco[df_loco['fold_id'] == fold_id]['cell_line'].iloc[0]
        trainval_cells = set(all_tnbc_cells) - {test_cell_f}
        trainval_df = tnbc_pathway[tnbc_pathway['ModelID'].isin(trainval_cells)]
        drug_means = trainval_df.groupby(drug_name_col)['LN_IC50'].mean().to_dict()
        mask = df_loco['fold_id'] == fold_id
        df_loco.loc[mask, 'y_mean'] = df_loco.loc[mask, 'drug_name'].map(lambda d: drug_means.get(d, np.nan))
    df_loco['y_mean'] = df_loco['y_mean'].fillna(df_loco.groupby('drug_name')['y_true'].transform('mean'))

    df_loco['y_true_resid'] = df_loco['y_true'] - df_loco['y_mean']
    df_loco['y_pred_resid'] = df_loco['y_pred'] - df_loco['y_mean']
    df_loco['y_pred_p1_resid'] = df_loco['y_pred_p1'] - df_loco['y_mean']
    if 'y_pred_p2' in df_loco.columns:
        df_loco['y_pred_p2_resid'] = df_loco['y_pred_p2'] - df_loco['y_mean']

    p_raw, _ = pearsonr(df_loco['y_true'], df_loco['y_pred'])
    r2_raw = r2_score(df_loco['y_true'], df_loco['y_pred'])
    p_resid, _ = pearsonr(df_loco['y_true_resid'], df_loco['y_pred_resid'])
    r2_resid = r2_score(df_loco['y_true_resid'], df_loco['y_pred_resid'])
    p_resid_p1, _ = pearsonr(df_loco['y_true_resid'], df_loco['y_pred_p1_resid'])
    r2_resid_p1 = r2_score(df_loco['y_true_resid'], df_loco['y_pred_p1_resid'])
    p_resid_p2 = pearsonr(df_loco['y_true_resid'], df_loco['y_pred_p2_resid'])[0] if 'y_pred_p2_resid' in df_loco.columns else np.nan
    r2_resid_p2 = r2_score(df_loco['y_true_resid'], df_loco['y_pred_p2_resid']) if 'y_pred_p2_resid' in df_loco.columns else np.nan

    per_drug = []
    for drug, g in df_loco.groupby('drug_name'):
        if len(g) >= 10:
            rho_m, _ = spearmanr(g['y_true'], g['y_pred'])
            rho_r, _ = spearmanr(g['y_true_resid'], g['y_pred_resid'])
            rho_r_p1, _ = spearmanr(g['y_true_resid'], g['y_pred_p1_resid'])
            rho_b, _ = spearmanr(g['y_true'], g['y_mean'])
            rec = {'drug': drug, 'n_cells': len(g), 'spearman_model': rho_m, 'spearman_resid': rho_r, 'spearman_resid_p1': rho_r_p1, 'spearman_baseline': rho_b}
            if 'y_pred_p2_resid' in g.columns:
                rho_r_p2, _ = spearmanr(g['y_true_resid'], g['y_pred_p2_resid'])
                rec['spearman_resid_p2'] = rho_r_p2
            per_drug.append(rec)
    df_drug = pd.DataFrame(per_drug)

    median_rho = np.nanmedian(df_drug['spearman_model'])
    median_rho_resid = np.nanmedian(df_drug['spearman_resid'])
    median_rho_resid_p1 = np.nanmedian(df_drug['spearman_resid_p1'])
    median_rho_resid_p2 = np.nanmedian(df_drug['spearman_resid_p2']) if 'spearman_resid_p2' in df_drug.columns else np.nan
    median_rho_base = np.nanmedian(df_drug['spearman_baseline'])
    frac_03 = (df_drug['spearman_model'] > 0.3).mean()
    frac_05 = (df_drug['spearman_model'] > 0.5).mean()

    df_loco.to_csv(viz_dir / "tnbc_gnn_loco_predictions.csv", index=False)
    metrics_path = viz_dir / "tnbc_loco_dreval_metrics.csv"
    drug_path = viz_dir / "tnbc_per_drug_spearman.csv"
    p_raw_p1, _ = pearsonr(df_loco['y_true'], df_loco['y_pred_p1'])
    r2_raw_p1 = r2_score(df_loco['y_true'], df_loco['y_pred_p1'])
    metrics_row = {'overall_pearson': p_raw, 'overall_r2': r2_raw, 'overall_pearson_p1': p_raw_p1, 'overall_r2_p1': r2_raw_p1, 'resid_pearson': p_resid, 'resid_r2': r2_resid, 'resid_pearson_p1': p_resid_p1, 'resid_r2_p1': r2_resid_p1, 'median_spearman': median_rho, 'median_spearman_resid': median_rho_resid, 'median_spearman_resid_p1': median_rho_resid_p1, 'median_baseline_spearman': median_rho_base, 'frac_rho_gt_03': frac_03, 'frac_rho_gt_05': frac_05}
    metrics_row['resid_pearson_p2'] = p_resid_p2
    metrics_row['resid_r2_p2'] = r2_resid_p2
    metrics_row['median_spearman_resid_p2'] = median_rho_resid_p2
    pd.DataFrame([metrics_row]).to_csv(metrics_path, index=False)
    df_drug.to_csv(drug_path, index=False)

In [ ]:
# Load LOCO CV results and model metric files
# All metrics (Phase 1/3 GNN, RF, ElasticNet) are from evaluation on TNBC held-out test cells (LOCO)
viz_dir = output_dir / "viz"

def load_loco_results():
    loco_path = output_dir / "loco_cv_results_cell_line.pkl"
    if not loco_path.exists():
        raise FileNotFoundError(f"LOCO results not found: {loco_path}. Run tnbc.ipynb LOCO CV first.")
    with open(loco_path, 'rb') as f:
        data = pickle.load(f)
    return data['fold_results']

fold_results = load_loco_results()
df_cell = pd.DataFrame([
    {'cell_line': r['test_cell'], 'P1_R2': r['phase1']['r2'], 'P2_R2': r['phase2']['r2'],
     'P3_R2': r['phase3']['r2']}
    for r in fold_results
])
df_cell = df_cell.sort_values('P3_R2', ascending=False).reset_index(drop=True)

# Build per-phase metric arrays for distribution plots
phase1_r2 = np.array([r['phase1']['r2'] for r in fold_results])
phase2_r2 = np.array([r['phase2']['r2'] for r in fold_results])
phase3_r2 = np.array([r['phase3']['r2'] for r in fold_results])

### Load Predictions

In [ ]:
# Prediction CSVs (single source for overall R²/Pearson - bar chart and boxplot)
df_gnn_pred = None
for p in [viz_dir / "tnbc_gnn_loco_predictions.csv", output_dir / "tnbc_gnn_loco_predictions.csv"]:
    if p.exists():
        tmp = pd.read_csv(p)
        if 'y_true' in tmp.columns and 'y_pred' in tmp.columns and 'fold_id' in tmp.columns:
            df_gnn_pred = tmp
            break
df_rf_pred = None
df_en_pred = None
for p in [output_dir / "tnbc_rf_loco_predictions.csv", output_dir / "rf_loco_predictions.csv"]:
    if p.exists():
        tmp = pd.read_csv(p)
        if 'y_true' in tmp.columns and 'y_pred' in tmp.columns:
            df_rf_pred = tmp
            break
for p in [output_dir / "tnbc_elasticnet_loco_predictions.csv", output_dir / "en_loco_predictions.csv"]:
    if p.exists():
        tmp = pd.read_csv(p)
        if 'y_true' in tmp.columns and 'y_pred' in tmp.columns:
            df_en_pred = tmp
            break

### Load Evaluation Metrics

In [ ]:
# Load model comparison metrics (overall_r2/pearson from prediction CSVs; resid/median from metric CSVs)
# Also load DrEval Phase 1 vs Phase 3 residual metrics and per-drug Spearman CSVs
def load_model_metrics():
    metrics_list = []
    dreval_path = viz_dir / "tnbc_loco_dreval_metrics.csv"
    if not dreval_path.exists():
        dreval_path = output_dir / "tnbc_loco_dreval_metrics.csv"
    if dreval_path.exists():
        d = pd.read_csv(dreval_path).iloc[0]
        p3_p = _mean_per_fold_pearson(df_gnn_pred) if df_gnn_pred is not None else d['overall_pearson']
        p3_r2 = _mean_per_fold_r2(df_gnn_pred) if df_gnn_pred is not None else d['overall_r2']
        if p3_p is None:
            p3_p = d['overall_pearson']
        if p3_r2 is None:
            p3_r2 = d['overall_r2']
        metrics_list.append(pd.DataFrame([{'model': 'Phase 3 GNN', 'overall_pearson': p3_p,
            'overall_r2': p3_r2, 'resid_pearson': d['resid_pearson'], 'resid_r2': d['resid_r2'],
            'median_spearman': d['median_spearman'], 'median_spearman_resid': d['median_spearman_resid'],
            'frac_rho_gt_03': d['frac_rho_gt_03'], 'frac_rho_gt_05': d['frac_rho_gt_05']}]))
        if 'resid_pearson_p1' in d.index:
            p1_pearson = d.get('overall_pearson_p1', np.nanmean([r['phase1']['pearson'] for r in fold_results]) if fold_results else np.nan)
            p1_r2 = d.get('overall_r2_p1', np.nanmean([r['phase1']['r2'] for r in fold_results]) if fold_results else np.nan)
            p1_median_spearman = np.nanmean([r['phase1']['spearman'] for r in fold_results]) if fold_results else np.nan
            p1_frac_03 = p1_frac_05 = np.nan
            drug_path = viz_dir / "tnbc_per_drug_spearman.csv"
            if not drug_path.exists():
                drug_path = output_dir / "tnbc_per_drug_spearman.csv"
            if drug_path.exists():
                df_drug = pd.read_csv(drug_path)
                if 'spearman_resid_p1' in df_drug.columns:
                    p1_frac_03 = (df_drug['spearman_resid_p1'] > 0.3).mean()
                    p1_frac_05 = (df_drug['spearman_resid_p1'] > 0.5).mean()
            metrics_list.append(pd.DataFrame([{'model': 'Phase 1 GNN', 'overall_pearson': p1_pearson,
                'overall_r2': p1_r2, 'resid_pearson': d['resid_pearson_p1'], 'resid_r2': d['resid_r2_p1'],
                'median_spearman': p1_median_spearman, 'median_spearman_resid': d['median_spearman_resid_p1'],
                'frac_rho_gt_03': p1_frac_03, 'frac_rho_gt_05': p1_frac_05}]))
    for path, model_name, pred_df in [
            (output_dir / "tnbc_rf_loco_metrics.csv", "Random Forest", df_rf_pred),
            (output_dir / "tnbc_elasticnet_loco_metrics.csv", "ElasticNet", df_en_pred)
        ]:
        if path.exists():
            m = pd.read_csv(path).iloc[0]
            p = _mean_per_fold_pearson(pred_df) if pred_df is not None else m.get('overall_pearson')
            r2 = _mean_per_fold_r2(pred_df) if pred_df is not None else m.get('overall_r2')
            if p is None:
                p = m.get('overall_pearson')
            if r2 is None:
                r2 = m.get('overall_r2')
            row = {k: m[k] for k in ['resid_pearson', 'resid_r2', 'median_spearman',
                'median_spearman_resid', 'frac_rho_gt_03', 'frac_rho_gt_05'] if k in m.index}
            metrics_list.append(pd.DataFrame([{'model': model_name, 'overall_pearson': p, 'overall_r2': r2, **row}]))
    if not metrics_list:
        missing_files = []
        if not (viz_dir / "tnbc_loco_dreval_metrics.csv").exists() and not (output_dir / "tnbc_loco_dreval_metrics.csv").exists():
            missing_files.append("tnbc_loco_dreval_metrics.csv")
        if not (output_dir / "tnbc_rf_loco_metrics.csv").exists():
            missing_files.append("tnbc_rf_loco_metrics.csv")
        if not (output_dir / "tnbc_elasticnet_loco_metrics.csv").exists():
            missing_files.append("tnbc_elasticnet_loco_metrics.csv")
        print(f"Warning: No metric files found. Missing: {', '.join(missing_files)}")
        print("Please run the DrEval, RF, and ElasticNet evaluation cells in tnbc.ipynb first.")
        return None
    df = pd.concat(metrics_list, ignore_index=True)
    order = ["Phase 1 GNN", "Phase 3 GNN", "Random Forest", "ElasticNet"]
    df['model'] = pd.Categorical(df['model'], categories=[m for m in order if m in df['model'].values], ordered=True)
    return df.sort_values('model').reset_index(drop=True)

df_metrics = load_model_metrics()

# DrEval Phase 1 vs Phase 3 residual metrics (GNN only)
dreval_p1_vs_p3 = None
_dreval = viz_dir / "tnbc_loco_dreval_metrics.csv"
if not _dreval.exists():
    _dreval = output_dir / "tnbc_loco_dreval_metrics.csv"
if _dreval.exists():
    d = pd.read_csv(_dreval).iloc[0]
    if 'resid_pearson_p1' in d.index:
        p1_rp = d['resid_pearson_p1']
        p1_rr2 = d['resid_r2_p1']
        p1_ms = d['median_spearman_resid_p1']
        p2_rp = d.get('resid_pearson_p2', np.nan)
        p2_rr2 = d.get('resid_r2_p2', np.nan)
        p2_ms = d.get('median_spearman_resid_p2', np.nan)
        dreval_p1_vs_p3 = {
            'Resid Pearson': (p1_rp, p2_rp, d['resid_pearson']),
            'Resid R²': (p1_rr2, p2_rr2, d['resid_r2']),
            'Median Spearman (resid)': (p1_ms, p2_ms, d['median_spearman_resid']),
        }

# Per-drug Spearman distributions (for fair metrics plot)
def load_per_drug_spearman():
    out = []
    for path, name in [
        (viz_dir / "tnbc_per_drug_spearman.csv" if (viz_dir / "tnbc_per_drug_spearman.csv").exists() else output_dir / "tnbc_per_drug_spearman.csv", "Phase 3 GNN"),
        (output_dir / "tnbc_rf_per_drug_spearman.csv", "Random Forest"),
        (output_dir / "tnbc_elasticnet_per_drug_spearman.csv", "ElasticNet"),
    ]:
        if path.exists():
            df = pd.read_csv(path)
            if 'spearman_resid' in df.columns:
                out.append(df.assign(model=name))
    return pd.concat(out, ignore_index=True) if out else None

df_per_drug = load_per_drug_spearman()

# Fix Phase 1 median_spearman: must use median per-drug (like P3/RF/EN), not overall from fold_results
if df_metrics is not None and df_gnn_pred is not None and not df_gnn_pred.empty:
    if 'y_pred_p1' in df_gnn_pred.columns and 'y_true' in df_gnn_pred.columns and 'drug_name' in df_gnn_pred.columns:
        p1_per_drug = []
        for drug, g in df_gnn_pred.groupby('drug_name'):
            if len(g) >= 10:
                r, _ = spearmanr(g['y_true'], g['y_pred_p1'])
                p1_per_drug.append(r)
        if p1_per_drug:
            p1_median = np.nanmedian(p1_per_drug)
            mask = df_metrics['model'] == 'Phase 1 GNN'
            if mask.any():
                df_metrics.loc[mask, 'median_spearman'] = p1_median

# Drug class (PUTATIVE_TARGET) from GDSC2 for optional scatter coloring
drug_to_class = None
gdsc_path = project_root / "data" / "raw" / "GDSC2 Fitted Dose Response Oct 27 2023.xlsx"
try:
    if gdsc_path.exists():
        cols = pd.read_excel(gdsc_path, nrows=0).columns
        if "PUTATIVE_TARGET" in cols and "DRUG_NAME" in cols:
            gdsc = pd.read_excel(gdsc_path, usecols=["DRUG_NAME", "PUTATIVE_TARGET"])
            drug_to_class = gdsc.drop_duplicates("DRUG_NAME").set_index("DRUG_NAME")["PUTATIVE_TARGET"].to_dict()
except Exception:
    pass

## 1. R² Frequency Distribution

Histogram of per-fold R² values across Phases 1, 2, and 3. Shows how fine-tuning progressively improves prediction accuracy on held-out TNBC cell lines.

In [ ]:
def plot_r2_frequency_distribution(phase1_r2, phase2_r2, phase3_r2, output_dir):
    """Line plot: R² bin vs Frequency (%) for each phase. Reference: monochrome, markers, distinct line styles."""
    bins = [0, 0.2, 0.4, 0.6, 0.8, 1.0]
    labels = ['0–0.2', '0.2–0.4', '0.4–0.6', '0.6–0.8', '0.8–1.0']
    x_vals = np.arange(len(labels))

    def bin_freq(vals):
        h, _ = np.histogram(vals, bins=bins)
        return 100 * h / (len(vals) or 1)

    freq1 = bin_freq(phase1_r2)
    freq2 = bin_freq(phase2_r2)
    freq3 = bin_freq(phase3_r2)

    fig, ax = plt.subplots(figsize=(7, 4), facecolor='white')
    ax.set_facecolor('white')
    ax.yaxis.grid(True, color='#C0C0C0', linestyle='--', linewidth=0.5)
    ax.xaxis.grid(True, color='#C0C0C0', linestyle='--', linewidth=0.5)
    ax.set_axisbelow(True)

    ax.plot(x_vals, freq1, 'k-', marker='o', label='Phase 1', markevery=1)
    ax.plot(x_vals, freq2, 'k--', marker='s', label='Phase 2', markevery=1)
    ax.plot(x_vals, freq3, 'k-.', marker='^', label='Phase 3', markevery=1)

    ax.set_xlabel('R² Range')
    ax.set_ylabel('Frequency (%)')
    ax.set_title('')
    ax.set_xticks(x_vals)
    ax.set_xticklabels(labels)
    ax.legend(loc=_best_legend_loc(ax), frameon=False)
    ax.set_ylim(0, None)
    plt.tight_layout()
    fig.savefig(output_dir / 'tnbc_r2_frequency.pdf', facecolor='white', bbox_inches='tight')
    fig.savefig(output_dir / 'tnbc_r2_frequency.png', facecolor='white', bbox_inches='tight')
    plt.show()

plot_r2_frequency_distribution(phase1_r2, phase2_r2, phase3_r2, output_dir)

## 2. Per-Phase Mean Metrics

Mean Pearson r, R², and RMSE across LOCO folds for each phase. Demonstrates the value of cascading transfer learning from pan-cancer → Breast → TNBC.

In [ ]:
def plot_per_phase_mean_metrics(fold_results, output_dir):
    """Per-phase correlation and error metrics as separate single-panel figures."""
    w = 0.2

    def _style(ax):
        ax.set_facecolor('white')
        ax.yaxis.grid(True, color='#C0C0C0', linestyle='--', linewidth=0.5)
        ax.set_axisbelow(True)

    # Correlation: Pearson, Spearman, R²
    cats1, keys1 = ['Pearson', 'Spearman', 'R²'], ['pearson', 'spearman', 'r2']
    data1 = {p: [np.nanmean([r[pk][k] for r in fold_results]) for k in keys1]
             for p, pk in [('Phase 1', 'phase1'), ('Phase 2', 'phase2'), ('Phase 3', 'phase3')]}
    x1 = np.arange(3)
    fig1, ax1 = plt.subplots(figsize=(5, 4), facecolor='white')
    _style(ax1)
    for i, (phase, hatch) in enumerate([('Phase 1', ''), ('Phase 2', '/'), ('Phase 3', 'xx')]):
        bars = ax1.bar(x1 + (i - 1) * w, data1[phase], width=w * 0.9, color='white', edgecolor='black',
               linewidth=0.8, hatch=hatch, label=phase)
        _label_bars(ax1, bars, data1[phase])
    ax1.set_xticks(x1)
    ax1.set_xticklabels(cats1)
    ax1.set_ylabel('Mean')
    ax1.set_title('')
    ax1.legend(loc=_best_legend_loc(ax1), frameon=False)
    ax1.set_ylim(0, 1.05)
    plt.tight_layout()
    fig1.savefig(output_dir / 'tnbc_per_phase_correlation.pdf', facecolor='white', bbox_inches='tight')
    fig1.savefig(output_dir / 'tnbc_per_phase_correlation.png', facecolor='white', bbox_inches='tight')
    plt.close()

    # Error: RMSE, MAE
    cats2, keys2 = ['RMSE', 'MAE'], ['rmse', 'mae']
    data2 = {p: [np.nanmean([r[pk][k] for r in fold_results]) for k in keys2]
             for p, pk in [('Phase 1', 'phase1'), ('Phase 2', 'phase2'), ('Phase 3', 'phase3')]}
    x2 = np.arange(2)
    fig2, ax2 = plt.subplots(figsize=(5, 4), facecolor='white')
    _style(ax2)
    for i, (phase, hatch) in enumerate([('Phase 1', ''), ('Phase 2', '/'), ('Phase 3', 'xx')]):
        bars = ax2.bar(x2 + (i - 1) * w, data2[phase], width=w * 0.9, color='white', edgecolor='black',
               linewidth=0.8, hatch=hatch, label=phase)
        _label_bars(ax2, bars, data2[phase])
    ax2.set_xticks(x2)
    ax2.set_xticklabels(cats2)
    ax2.set_ylabel('Mean')
    ax2.set_title('')
    ax2.legend(loc=_best_legend_loc(ax2), frameon=False)
    plt.tight_layout()
    fig2.savefig(output_dir / 'tnbc_per_phase_error.pdf', facecolor='white', bbox_inches='tight')
    fig2.savefig(output_dir / 'tnbc_per_phase_error.png', facecolor='white', bbox_inches='tight')
    plt.close()

plot_per_phase_mean_metrics(fold_results, output_dir)

## 3. R² Distribution — Box Plots

Box plots of per-fold R² for (a) phases 1/2/3 and (b) GNN vs Random Forest vs ElasticNet baselines.

In [ ]:
def plot_r2_boxplot(phase1_r2, phase2_r2, phase3_r2, output_dir):
    """Box plot: R² distribution per phase. Reference: hatching, light grid."""
    fig, ax = plt.subplots(figsize=(6, 4), facecolor='white')
    ax.set_facecolor('white')
    ax.yaxis.grid(True, color='#C0C0C0', linestyle='--', linewidth=0.5)
    ax.set_axisbelow(True)

    data = [phase1_r2, phase2_r2, phase3_r2]
    labels = ['Phase 1', 'Phase 2', 'Phase 3']
    hatches = ['', '/', 'xx']
    bp = ax.boxplot(data, labels=labels, patch_artist=True, widths=0.6)
    for i, (patch, h) in enumerate(zip(bp['boxes'], hatches)):
        patch.set_facecolor('white')
        patch.set_edgecolor('black')
        patch.set_hatch(h)
    for el in bp['medians'] + bp['whiskers'] + bp['caps']:
        el.set_color('black')
    ax.set_ylabel('R²')
    ax.set_title('')
    ax.set_ylim(-0.05, 1.05)
    plt.tight_layout()
    fig.savefig(output_dir / 'tnbc_r2_boxplot.pdf', facecolor='white', bbox_inches='tight')
    fig.savefig(output_dir / 'tnbc_r2_boxplot.png', facecolor='white', bbox_inches='tight')
    plt.show()

plot_r2_boxplot(phase1_r2, phase2_r2, phase3_r2, output_dir)

In [ ]:
def plot_r2_boxplot_models(df_gnn_pred, df_rf_pred, df_en_pred, output_dir):
    """Box plot: R² distribution per model (P3 GNN, RF, EN). Same source as bar chart: prediction CSVs."""
    def _per_fold_r2(df):
        if df is None or df.empty or 'fold_id' not in df.columns:
            return np.array([])
        r2_list = []
        for _, g in df.groupby('fold_id'):
            if len(g) >= 2:
                r2_list.append(r2_score(g['y_true'], g['y_pred']))
        return np.array(r2_list)
    data = [_per_fold_r2(df_gnn_pred), _per_fold_r2(df_rf_pred), _per_fold_r2(df_en_pred)]
    labels = ['Phase 3 GNN', 'Random Forest', 'ElasticNet']
    data = [d if len(d) > 0 else np.array([np.nan]) for d in data]
    valid = [d for d in data if np.any(np.isfinite(d))]
    if not valid:
        return
    fig, ax = plt.subplots(figsize=(6, 4), facecolor='white')
    ax.set_facecolor('white')
    ax.yaxis.grid(True, color='#C0C0C0', linestyle='--', linewidth=0.5)
    ax.set_axisbelow(True)
    hatches = ['', '/', 'xx']
    bp = ax.boxplot(data, labels=labels, patch_artist=True, widths=0.6)
    for i, (patch, h) in enumerate(zip(bp['boxes'], hatches)):
        patch.set_facecolor('white')
        patch.set_edgecolor('black')
        patch.set_hatch(h)
    for el in bp['medians'] + bp['whiskers'] + bp['caps']:
        el.set_color('black')
    ax.set_ylabel('R²')
    ax.set_title('')
    ax.set_ylim(-0.05, 1.05)
    plt.tight_layout()
    fig.savefig(output_dir / 'tnbc_r2_boxplot_models.pdf', facecolor='white', bbox_inches='tight')
    fig.savefig(output_dir / 'tnbc_r2_boxplot_models.png', facecolor='white', bbox_inches='tight')
    plt.show()

plot_r2_boxplot_models(df_gnn_pred, df_rf_pred, df_en_pred, output_dir)

## 4. R² per Cell Line

Per-cell-line R² from LOCO CV. Identifies which TNBC cell lines are well-predicted versus difficult to model.

In [ ]:
def plot_r2_per_cell(df_cell, output_dir):
    """Horizontal bar: cell line vs Phase 3 R². Reference: monochrome, clean."""
    fig, ax = plt.subplots(figsize=(8, max(5, len(df_cell) * 0.35)), facecolor='white')
    ax.set_facecolor('white')
    ax.grid(True, axis='x', color='#C0C0C0', linestyle='--', linewidth=0.5)
    ax.set_axisbelow(True)

    y_pos = np.arange(len(df_cell))
    bars = ax.barh(y_pos, df_cell['P3_R2'], height=0.7, color='white', edgecolor='black', linewidth=0.8)
    median_r2 = df_cell['P3_R2'].median()
    ax.axvline(median_r2, color='black', linestyle='--', linewidth=1, label=f'Median R² = {median_r2:.3f}')
    ax.set_yticks(y_pos)
    ax.set_yticklabels(df_cell['cell_line'])
    ax.set_xlabel('Phase 3 R²')
    ax.set_ylabel('Cell Line (Held-Out Test)')
    ax.set_title('')
    ax.legend(loc=_best_legend_loc(ax), frameon=False)
    ax.invert_yaxis()
    ax.set_xlim(0, 1.05)
    plt.tight_layout()
    fig.savefig(output_dir / 'tnbc_r2_per_cell.pdf', facecolor='white', bbox_inches='tight')
    fig.savefig(output_dir / 'tnbc_r2_per_cell.png', facecolor='white', bbox_inches='tight')
    plt.show()

plot_r2_per_cell(df_cell, output_dir)

## 5. Model Comparison

Overall and residualised metrics for Phase 3 GNN, Phase 1 GNN, Random Forest, and ElasticNet — all evaluated on the same LOCO held-out TNBC cells.

In [ ]:
def plot_model_comparison(df_metrics, output_dir, df_gnn_pred=None, df_rf_pred=None, df_en_pred=None):
    """Grouped bar: model comparison on TNBC LOCO test. Reference: hatching, clean layout."""
    if df_metrics is None:
        return
    def _safe_max(v):
        v = np.asarray(v)
        return float(np.nanmax(v)) if np.any(np.isfinite(v)) else 0.1

    def _per_fold_vals(model_name):
        """Per-fold Pearson and R² from prediction df; return (p_arr, r2_arr)."""
        pred_map = {'Phase 3 GNN': df_gnn_pred, 'Random Forest': df_rf_pred, 'ElasticNet': df_en_pred}
        df_p = pred_map.get(model_name)
        if df_p is None or df_p.empty or 'fold_id' not in df_p.columns or 'y_true' not in df_p.columns or 'y_pred' not in df_p.columns:
            return np.array([]), np.array([])
        p_vals, r2_vals = [], []
        for _, g in df_p.groupby('fold_id'):
            if len(g) < 2:
                continue
            r, _ = pearsonr(g['y_true'], g['y_pred'])
            p_vals.append(r)
            r2_vals.append(r2_score(g['y_true'], g['y_pred']))
        return np.array(p_vals), np.array(r2_vals)

    # 1) Overall Pearson & R² (P3, RF, EN only; bar chart)
    df_overall = df_metrics[df_metrics['model'] != 'Phase 1 GNN'].reset_index(drop=True)
    if df_overall.empty:
        return
    x = np.arange(len(df_overall))
    w = 0.65
    p_vals = df_overall['overall_pearson'].fillna(0).values
    r2_vals = df_overall['overall_r2'].fillna(0).values
    fig, ax = plt.subplots(figsize=(5, 4), facecolor='white')
    ax.set_facecolor('white')
    ax.yaxis.grid(True, color='#C0C0C0', linestyle='--', linewidth=0.5)
    ax.set_axisbelow(True)
    b1 = ax.bar(x - w/4, p_vals, width=w/2, color='white', edgecolor='black', linewidth=0.8, label='Pearson')
    b2 = ax.bar(x + w/4, r2_vals, width=w/2, color='white', edgecolor='black', hatch='/', linewidth=0.8, label='R²')
    _label_bars(ax, b1, p_vals)
    _label_bars(ax, b2, r2_vals)
    ax.set_xticks(x)
    ax.set_xticklabels(df_overall['model'], rotation=25, ha='right')
    ax.set_ylabel('Correlation')
    ax.set_title('')
    ax.legend(loc='upper center', ncol=2, frameon=False, bbox_to_anchor=(0.5, 1.02))
    y_top = max(_safe_max(df_overall['overall_pearson']), _safe_max(df_overall['overall_r2']), 0.01) * 1.15
    ax.set_ylim(0, y_top)
    plt.tight_layout()
    fig.savefig(output_dir / 'tnbc_model_overall.pdf', facecolor='white', bbox_inches='tight')
    fig.savefig(output_dir / 'tnbc_model_overall.png', facecolor='white', bbox_inches='tight')
    plt.close()

    # 1b) Overall Pearson & R² distribution (P3, RF, EN only; boxplot)
    models = df_overall['model'].tolist()
    p_data = [_per_fold_vals(m)[0] for m in models]
    r2_data = [_per_fold_vals(m)[1] for m in models]
    p_data = [d if len(d) > 0 else np.array([np.nan]) for d in p_data]
    r2_data = [d if len(d) > 0 else np.array([np.nan]) for d in r2_data]
    fig, ax = plt.subplots(figsize=(6, 4), facecolor='white')
    ax.set_facecolor('white')
    ax.yaxis.grid(True, color='#C0C0C0', linestyle='--', linewidth=0.5)
    ax.set_axisbelow(True)
    n = len(models)
    xbp = np.arange(n)
    wbp = 0.35
    positions_p = xbp - wbp/2
    positions_r2 = xbp + wbp/2
    bp = ax.boxplot(p_data, positions=positions_p, widths=wbp*0.85, patch_artist=True)
    for patch in bp['boxes']:
        patch.set_facecolor('white')
        patch.set_edgecolor('black')
    for el in bp['medians'] + bp['whiskers'] + bp['caps'] + bp['fliers']:
        el.set_color('black')
    bp2 = ax.boxplot(r2_data, positions=positions_r2, widths=wbp*0.85, patch_artist=True)
    for patch in bp2['boxes']:
        patch.set_facecolor('white')
        patch.set_edgecolor('black')
        patch.set_hatch('/')
    for el in bp2['medians'] + bp2['whiskers'] + bp2['caps'] + bp2['fliers']:
        el.set_color('black')
    handles = [Patch(facecolor='white', edgecolor='black'), Patch(facecolor='white', edgecolor='black', hatch='/')]
    ax.legend(handles=handles, labels=['Pearson', 'R²'], loc='upper center', ncol=2, frameon=False, bbox_to_anchor=(0.5, 1.02))
    ax.set_xticks(xbp)
    ax.set_xticklabels(models, rotation=25, ha='right')
    ax.set_ylabel('Correlation')
    all_vals = np.concatenate([d[np.isfinite(d)] for d in p_data + r2_data])
    y_hi = np.nanmax(all_vals) * 1.15 if len(all_vals) > 0 else 1.0
    ax.set_ylim(0, max(y_hi, 0.05))
    plt.tight_layout()
    fig.savefig(output_dir / 'tnbc_model_overall_distribution.pdf', facecolor='white', bbox_inches='tight')
    fig.savefig(output_dir / 'tnbc_model_overall_distribution.png', facecolor='white', bbox_inches='tight')
    plt.close()

    # 2) Residualized Pearson & R² (all models including P1)
    x = np.arange(len(df_metrics))
    fig, ax = plt.subplots(figsize=(5, 4), facecolor='white')
    ax.set_facecolor('white')
    ax.yaxis.grid(True, color='#C0C0C0', linestyle='--', linewidth=0.5)
    ax.set_axisbelow(True)
    rpr = df_metrics['resid_pearson'].fillna(0).values
    rr2 = df_metrics['resid_r2'].fillna(0).values
    b1 = ax.bar(x - w/4, rpr, width=w/2, color='white', edgecolor='black', linewidth=0.8, label='Resid Pearson')
    b2 = ax.bar(x + w/4, rr2, width=w/2, color='white', edgecolor='black', hatch='/', linewidth=0.8, label='Resid R²')
    _label_bars(ax, b1, rpr)
    _label_bars(ax, b2, rr2)
    ax.set_xticks(x)
    ax.set_xticklabels(df_metrics['model'], rotation=25, ha='right')
    ax.set_ylabel('Correlation')
    ax.set_title('')
    ax.legend(loc=_best_legend_loc(ax), frameon=False)
    y_hi = max(_safe_max(rpr), _safe_max(rr2), 0.01) * 1.15
    y_lo = min(0, np.nanmin(rpr), np.nanmin(rr2)) - 0.05
    ax.set_ylim(y_lo, y_hi)

    plt.tight_layout()
    fig.savefig(output_dir / 'tnbc_model_residualized.pdf', facecolor='white', bbox_inches='tight')
    fig.savefig(output_dir / 'tnbc_model_residualized.png', facecolor='white', bbox_inches='tight')
    plt.close()

    # 3) Median per-drug Spearman
    fig, ax = plt.subplots(figsize=(5, 4), facecolor='white')
    ax.set_facecolor('white')
    ax.yaxis.grid(True, color='#C0C0C0', linestyle='--', linewidth=0.5)
    ax.set_axisbelow(True)
    rho = df_metrics['median_spearman'].fillna(0).values
    rho_r = df_metrics['median_spearman_resid'].fillna(0).values
    b1 = ax.bar(x - w/4, rho, width=w/2, color='white', edgecolor='black', linewidth=0.8, label='Spearman')
    b2 = ax.bar(x + w/4, rho_r, width=w/2, color='white', edgecolor='black', hatch='/', linewidth=0.8, label='Spearman (resid)')
    _label_bars(ax, b1, rho)
    _label_bars(ax, b2, rho_r)
    ax.set_xticks(x)
    ax.set_xticklabels(df_metrics['model'], rotation=25, ha='right')
    ax.set_ylabel('Spearman ρ')
    ax.set_title('')
    ax.legend(loc=_best_legend_loc(ax), frameon=False)
    rho_min = min(np.nanmin(rho), np.nanmin(rho_r), -0.1)
    rho_max = max(np.nanmax(rho), np.nanmax(rho_r), 0.01) * 1.15
    ax.set_ylim(rho_min - 0.05, rho_max)

    plt.tight_layout()
    fig.savefig(output_dir / 'tnbc_model_median_spearman.pdf', facecolor='white', bbox_inches='tight')
    fig.savefig(output_dir / 'tnbc_model_median_spearman.png', facecolor='white', bbox_inches='tight')
    plt.close()

    # 4) Fraction drugs ρ > threshold
    fig, ax = plt.subplots(figsize=(5, 4), facecolor='white')
    ax.set_facecolor('white')
    ax.yaxis.grid(True, color='#C0C0C0', linestyle='--', linewidth=0.5)
    ax.set_axisbelow(True)
    f03 = df_metrics['frac_rho_gt_03'].fillna(0).values
    f05 = df_metrics['frac_rho_gt_05'].fillna(0).values
    b1 = ax.bar(x - w/4, f03, width=w/2, color='white', edgecolor='black', linewidth=0.8, label='ρ > 0.3')
    b2 = ax.bar(x + w/4, f05, width=w/2, color='white', edgecolor='black', hatch='/', linewidth=0.8, label='ρ > 0.5')
    _label_bars(ax, b1, f03)
    _label_bars(ax, b2, f05)
    ax.set_xticks(x)
    ax.set_xticklabels(df_metrics['model'], rotation=25, ha='right')
    ax.set_ylabel('Fraction of Drugs')
    ax.set_title('')
    ax.legend(loc=_best_legend_loc(ax), frameon=False)
    ax.set_ylim(0, 1)

    plt.tight_layout()
    fig.savefig(output_dir / 'tnbc_model_frac_drugs.pdf', facecolor='white', bbox_inches='tight')
    fig.savefig(output_dir / 'tnbc_model_frac_drugs.png', facecolor='white', bbox_inches='tight')
    plt.close()

plot_model_comparison(df_metrics, output_dir, df_gnn_pred=df_gnn_pred, df_rf_pred=df_rf_pred, df_en_pred=df_en_pred)

## 6. Phase 1 vs Phase 3 GNN (Residualised)

Residualised Pearson and R² strip the drug-mean baseline to isolate each model's ability to distinguish cell-line-specific sensitivity beyond trivial drug potency.

In [ ]:
def plot_p1_vs_p3_residual(dreval_p1_vs_p3, output_dir):
    """Grouped bar: Phase 1 vs Phase 2 vs Phase 3 residual metrics. Fair comparison (drug-mean stripped)."""
    if dreval_p1_vs_p3 is None:
        return
    categories = list(dreval_p1_vs_p3.keys())
    x = np.arange(len(categories))
    w = 0.25
    p1_vals = [dreval_p1_vs_p3[k][0] for k in categories]
    p2_vals = [dreval_p1_vs_p3[k][1] for k in categories]
    p3_vals = [dreval_p1_vs_p3[k][2] for k in categories]

    fig, ax = plt.subplots(figsize=(6, 4), facecolor='white')
    ax.set_facecolor('white')
    ax.yaxis.grid(True, color='#C0C0C0', linestyle='--', linewidth=0.5)
    ax.set_axisbelow(True)

    b1 = ax.bar(x - w, p1_vals, width=w * 0.9, color='white', edgecolor='black', hatch='/', linewidth=0.8, label='Phase 1')
    b2 = ax.bar(x, p2_vals, width=w * 0.9, color='white', edgecolor='black', hatch='', linewidth=0.8, label='Phase 2')
    b3 = ax.bar(x + w, p3_vals, width=w * 0.9, color='white', edgecolor='black', hatch='xx', linewidth=0.8, label='Phase 3')
    _label_bars(ax, b1, p1_vals)
    _label_bars(ax, b2, p2_vals)
    _label_bars(ax, b3, p3_vals)
    ax.axhline(0, color='black', linewidth=0.5)
    ax.set_xticks(x)
    ax.set_xticklabels(categories, rotation=20, ha='right')
    ax.set_ylabel('Value')
    ax.set_title('')
    ax.legend(loc=_best_legend_loc(ax), frameon=False)
    all_vals = [v for v in p1_vals + p2_vals + p3_vals if np.isfinite(v)]
    y_min = (min(all_vals) - 0.05) if all_vals else -0.1
    y_max = (max(all_vals) + 0.05) if all_vals else 0.1
    ax.set_ylim(y_min, y_max)
    plt.tight_layout()
    fig.savefig(output_dir / 'tnbc_p1_vs_p3_residual.pdf', facecolor='white', bbox_inches='tight')
    fig.savefig(output_dir / 'tnbc_p1_vs_p3_residual.png', facecolor='white', bbox_inches='tight')
    plt.show()

plot_p1_vs_p3_residual(dreval_p1_vs_p3, output_dir)

## 7. Per-Drug Spearman Distribution

Distribution of per-drug Spearman ρ across all models. Measures each model's ability to rank cell lines by sensitivity within individual drugs.

In [ ]:
# Classification Head Performance (Phase 3 GNN)
# Sensitive (1) vs resistant (0) from per-drug median threshold
gnn_path = viz_dir / "tnbc_gnn_loco_predictions.csv"
if not gnn_path.exists():
    gnn_path = output_dir / "tnbc_gnn_loco_predictions.csv"
if gnn_path.exists():
    df_cls = pd.read_csv(gnn_path)
    if 'y_true_class' in df_cls.columns and 'y_pred_class' in df_cls.columns:
        correct = (df_cls['y_true_class'] == df_cls['y_pred_class']).sum()
        wrong = len(df_cls) - correct
        tp = ((df_cls['y_true_class'] == 1) & (df_cls['y_pred_class'] == 1)).sum()
        tn = ((df_cls['y_true_class'] == 0) & (df_cls['y_pred_class'] == 0)).sum()
        fp = ((df_cls['y_true_class'] == 0) & (df_cls['y_pred_class'] == 1)).sum()
        fn = ((df_cls['y_true_class'] == 1) & (df_cls['y_pred_class'] == 0)).sum()
        acc = correct / len(df_cls) if len(df_cls) > 0 else 0.0
        print("=" * 50)
        print("Classification Head (Phase 3 GNN)")
        print("=" * 50)
        print(f"Correct: {correct}  |  Wrong: {wrong}  |  Total: {len(df_cls)}")
        print(f"TP: {tp}  TN: {tn}  FP: {fp}  FN: {fn}")
        print(f"Accuracy: {acc:.4f}")
        print("=" * 50)
    else:
        print("Classification columns (y_true_class, y_pred_class) missing. Re-run the GNN predictions cell.")
else:
    print("tnbc_gnn_loco_predictions.csv not found. Run the GNN predictions cell first.")

In [ ]:
def plot_per_drug_spearman_distribution(df_per_drug, output_dir):
    """Box plot: per-drug spearman_resid distribution by model."""
    if df_per_drug is None or df_per_drug.empty:
        return
    order = ["Phase 1 GNN", "Phase 3 GNN", "Random Forest", "ElasticNet"]
    df = df_per_drug[df_per_drug['model'].isin(order)].copy()
    df['model'] = pd.Categorical(df['model'], categories=order, ordered=True)
    df = df.sort_values('model')

    fig, ax = plt.subplots(figsize=(6, 4), facecolor='white')
    ax.set_facecolor('white')
    ax.yaxis.grid(True, color='#C0C0C0', linestyle='--', linewidth=0.5)
    ax.set_axisbelow(True)

    hatches = {'Phase 1 GNN': '/', 'Phase 3 GNN': '', 'Random Forest': 'xx', 'ElasticNet': '..'}
    bp = ax.boxplot(
        [df[df['model'] == m]['spearman_resid'].dropna().values for m in order],
        labels=order,
        patch_artist=True,
        widths=0.6
    )
    for i, (patch, m) in enumerate(zip(bp['boxes'], order)):
        patch.set_facecolor('white')
        patch.set_edgecolor('black')
        patch.set_hatch(hatches.get(m, ''))
    for el in bp['medians'] + bp['whiskers'] + bp['caps']:
        el.set_color('black')
    ax.axhline(0, color='black', linewidth=0.5, linestyle='--')
    ax.set_ylabel('Per-Drug Spearman ρ (Residualized)')
    ax.set_title('')
    plt.tight_layout()
    fig.savefig(output_dir / 'tnbc_per_drug_spearman_dist.pdf', facecolor='white', bbox_inches='tight')
    fig.savefig(output_dir / 'tnbc_per_drug_spearman_dist.png', facecolor='white', bbox_inches='tight')
    plt.show()

plot_per_drug_spearman_distribution(df_per_drug, output_dir)

## 8. Overall vs Residualised Metrics

Side-by-side comparison of raw and residualised (drug-mean-corrected) Pearson r and R² for all models. Quantifies how much of each model's performance is explained by drug potency versus cell-line-specific signal.

In [ ]:
def plot_overall_vs_residualized_pearson(df_metrics, output_dir):
    """Grouped bar: Overall vs Residualized Pearson per model."""
    if df_metrics is None:
        return
    x = np.arange(len(df_metrics))
    w = 0.2

    fig, ax = plt.subplots(figsize=(6, 4), facecolor='white')
    ax.set_facecolor('white')
    ax.yaxis.grid(True, color='#C0C0C0', linestyle='--', linewidth=0.5)
    ax.set_axisbelow(True)

    b1 = ax.bar(x - w, df_metrics['overall_pearson'].values, width=w * 0.9, color='white',
            edgecolor='black', linewidth=0.8, label='Overall Pearson')
    b2 = ax.bar(x, df_metrics['resid_pearson'].values, width=w * 0.9, color='white',
            edgecolor='black', hatch='/', linewidth=0.8, label='Resid Pearson')
    _label_bars(ax, b1, df_metrics['overall_pearson'].values)
    _label_bars(ax, b2, df_metrics['resid_pearson'].values)
    ax.set_xticks(x)
    ax.set_xticklabels(df_metrics['model'], rotation=25, ha='right')
    ax.set_ylabel('Pearson r')
    ax.set_title('')
    ax.legend(loc=_best_legend_loc(ax), frameon=False)
    ax.axhline(0, color='black', linewidth=0.5)
    y_lo = min(np.nanmin(df_metrics['overall_pearson'].values), np.nanmin(df_metrics['resid_pearson'].values), 0) - 0.05
    ax.set_ylim(y_lo, 1.05)

    plt.tight_layout()
    fig.savefig(output_dir / 'tnbc_overall_vs_residualized_pearson.pdf', facecolor='white', bbox_inches='tight')
    fig.savefig(output_dir / 'tnbc_overall_vs_residualized_pearson.png', facecolor='white', bbox_inches='tight')
    plt.close()

def plot_overall_vs_residualized_r2(df_metrics, output_dir):
    """Grouped bar: Overall vs Residualized R² per model."""
    if df_metrics is None:
        return
    x = np.arange(len(df_metrics))
    w = 0.2

    fig, ax = plt.subplots(figsize=(6, 4), facecolor='white')
    ax.set_facecolor('white')
    ax.yaxis.grid(True, color='#C0C0C0', linestyle='--', linewidth=0.5)
    ax.set_axisbelow(True)

    b1 = ax.bar(x - w, df_metrics['overall_r2'].values, width=w * 0.9, color='white',
            edgecolor='black', linewidth=0.8, label='Overall R²')
    b2 = ax.bar(x, df_metrics['resid_r2'].values, width=w * 0.9, color='white',
            edgecolor='black', hatch='/', linewidth=0.8, label='Resid R²')
    _label_bars(ax, b1, df_metrics['overall_r2'].values)
    _label_bars(ax, b2, df_metrics['resid_r2'].values)
    ax.set_xticks(x)
    ax.set_xticklabels(df_metrics['model'], rotation=25, ha='right')
    ax.set_ylabel('R²')
    ax.set_title('')
    ax.legend(loc=_best_legend_loc(ax), frameon=False)
    ax.axhline(0, color='black', linewidth=0.5)
    y_lo = min(df_metrics['overall_r2'].min(), df_metrics['resid_r2'].min(), 0) - 0.1
    ax.set_ylim(y_lo, 1.05)

    plt.tight_layout()
    fig.savefig(output_dir / 'tnbc_overall_vs_residualized_r2.pdf', facecolor='white', bbox_inches='tight')
    fig.savefig(output_dir / 'tnbc_overall_vs_residualized_r2.png', facecolor='white', bbox_inches='tight')
    plt.close()

### Execute Overall vs Residualised Plots

In [ ]:
viz_dir = output_dir / "viz"
metrics_path = viz_dir / "tnbc_loco_dreval_metrics.csv"
if not metrics_path.exists():
    metrics_path = output_dir / "tnbc_loco_dreval_metrics.csv"
if not metrics_path.exists():
    pred_path = viz_dir / "tnbc_gnn_loco_predictions.csv"
    if not pred_path.exists():
        pred_path = output_dir / "tnbc_gnn_loco_predictions.csv"
    if not pred_path.exists():
        raise RuntimeError("tnbc_gnn_loco_predictions.csv missing. Run tnbc.ipynb or the GNN predictions cell.")

    with open(output_dir / "loco_cv_results_cell_line.pkl", 'rb') as f:
        loco_data = pickle.load(f)
    _fr = loco_data['fold_results']
    all_tnbc_cells = sorted(set(r['test_cell'] for r in _fr))

    data_dir = project_root / "data" / "raw"
    gdsc2_df = pd.read_excel(data_dir / "GDSC2 Fitted Dose Response Oct 27 2023.xlsx")
    model_df = pd.read_csv(data_dir / "DepMap Model Data.csv")
    pathway_scores_raw = pd.read_csv(data_dir / "cell_ge.csv", index_col=0)
    pathway_names = pathway_scores_raw.index.tolist()
    pathway_data = pathway_scores_raw.T.reset_index()
    pathway_data.columns = ['CellLineName'] + pathway_names
    rmse_col = [col for col in gdsc2_df.columns if 'RMSE' in col.upper()][0]
    gdsc2_filtered = gdsc2_df[gdsc2_df[rmse_col] < 0.3].copy()
    drug_response = gdsc2_filtered[['DRUG_NAME', 'CELL_LINE_NAME', 'LN_IC50', 'COSMIC_ID']].copy()
    drug_response.columns = ['DrugName', 'CellLineName', 'LN_IC50', 'COSMICID']
    cell_name_to_modelid = dict(zip(model_df['StrippedCellLineName'].str.upper().str.replace('-', '').str.replace('_', ''), model_df['ModelID']))
    cosmic_to_modelid = model_df.drop_duplicates(subset='COSMICID', keep='first').set_index('COSMICID')['ModelID'].to_dict()
    pathway_data['ModelID'] = pathway_data['CellLineName'].apply(lambda x: cell_name_to_modelid.get(str(x).upper().replace('-', '').replace('_', ''), None))
    pathway_data = pathway_data[pathway_data['ModelID'].notna()].copy()
    drug_response['ModelID'] = drug_response['COSMICID'].apply(lambda x: cosmic_to_modelid.get(x, None))
    drug_response.loc[drug_response['ModelID'].isna(), 'ModelID'] = drug_response.loc[drug_response['ModelID'].isna(), 'CellLineName'].apply(lambda x: cell_name_to_modelid.get(str(x).upper().replace('-', '').replace('_', ''), None))
    pan_cancer_pathway = pathway_data.merge(drug_response[['DrugName', 'ModelID', 'LN_IC50']], on='ModelID', how='inner')
    pan_cancer_pathway = pan_cancer_pathway.rename(columns={'DrugName': 'DRUG_NAME'})
    model_cols = ['ModelID', 'StrippedCellLineName', 'OncotreeLineage', 'OncotreePrimaryDisease', 'OncotreeSubtype']
    for col in ['PatientSubtypeFeatures', 'ModelSubtypeFeatures']:
        if col in model_df.columns:
            model_cols.append(col)
    pan_cancer_pathway = pan_cancer_pathway.merge(model_df[model_cols], on='ModelID', how='left')
    mutation_file = data_dir / "Omics Somatic Mutations.csv"
    if not mutation_file.exists():
        mutation_file = data_dir / "OmicsSomaticMutations.csv"
    mutations_df = pd.read_csv(mutation_file, low_memory=False)
    priority_genes = ['BRCA1', 'BRCA2', 'TP53', 'PIK3CA', 'PTEN', 'RB1', 'EGFR', 'MYC', 'CDKN2A', 'GATA3', 'MAP3K1', 'KRAS', 'ARID1A', 'NF1', 'ERBB2', 'ESR1', 'PGR', 'AKT1']
    gene_col = 'HugoSymbol' if 'HugoSymbol' in mutations_df.columns else 'Hugo_Symbol'
    if 'LikelyLoF' in mutations_df.columns:
        mutations_df['is_mutated'] = mutations_df['LikelyLoF'].fillna(False).astype(int)
    elif 'TranscriptLikelyLof' in mutations_df.columns:
        mutations_df['is_mutated'] = mutations_df['TranscriptLikelyLof'].fillna(False).astype(int)
    elif 'VepImpact' in mutations_df.columns:
        mutations_df['is_mutated'] = mutations_df['VepImpact'].isin(['HIGH', 'MODERATE']).astype(int)
    else:
        mutations_df['is_mutated'] = 1
    mutation_matrix = mutations_df[mutations_df[gene_col].isin(priority_genes)].pivot_table(index='ModelID', columns=gene_col, values='is_mutated', aggfunc='max', fill_value=0)
    mutation_matrix = mutation_matrix.reindex(columns=priority_genes, fill_value=0)
    pan_cancer_pathway = pan_cancer_pathway.merge(mutation_matrix.reset_index(), on='ModelID', how='left')
    pan_cancer_pathway[priority_genes] = pan_cancer_pathway[priority_genes].fillna(0)
    cfg = {'phase2_lineage': 'Breast', 'phase3_subtype_col': 'ModelSubtypeFeatures', 'phase3_subtype_kw': 'TNBC'}
    breast_pathway = pan_cancer_pathway[pan_cancer_pathway['OncotreeLineage'] == cfg['phase2_lineage']].copy()
    _sub_col = cfg['phase3_subtype_col']
    _sub_kw = cfg['phase3_subtype_kw']
    if _sub_col in breast_pathway.columns:
        subtype_mask = breast_pathway[_sub_col].astype(str).str.contains(_sub_kw, case=False, na=False)
    else:
        psf = breast_pathway['PatientSubtypeFeatures'].astype(str).str.contains(_sub_kw, case=False, na=False) if 'PatientSubtypeFeatures' in breast_pathway.columns else pd.Series(False, index=breast_pathway.index)
        msf = breast_pathway['ModelSubtypeFeatures'].astype(str).str.contains(_sub_kw, case=False, na=False) if 'ModelSubtypeFeatures' in breast_pathway.columns else pd.Series(False, index=breast_pathway.index)
        subtype_mask = psf | msf
    tnbc_pathway = breast_pathway[subtype_mask].copy()

    df_loco = pd.read_csv(pred_path)

    df_loco['y_mean'] = np.nan
    for fold_id in df_loco['fold_id'].unique():
        test_cell_f = df_loco[df_loco['fold_id'] == fold_id]['cell_line'].iloc[0]
        trainval_cells = set(all_tnbc_cells) - {test_cell_f}
        trainval_df = tnbc_pathway[tnbc_pathway['ModelID'].isin(trainval_cells)]
        drug_means = trainval_df.groupby('DRUG_NAME')['LN_IC50'].mean().to_dict()
        mask = df_loco['fold_id'] == fold_id
        df_loco.loc[mask, 'y_mean'] = df_loco.loc[mask, 'drug_name'].map(lambda d: drug_means.get(d, np.nan))
    df_loco['y_mean'] = df_loco['y_mean'].fillna(df_loco.groupby('drug_name')['y_true'].transform('mean'))

    df_loco['y_true_resid'] = df_loco['y_true'] - df_loco['y_mean']
    df_loco['y_pred_resid'] = df_loco['y_pred'] - df_loco['y_mean']
    df_loco['y_pred_p1_resid'] = df_loco['y_pred_p1'] - df_loco['y_mean']
    if 'y_pred_p2' in df_loco.columns:
        df_loco['y_pred_p2_resid'] = df_loco['y_pred_p2'] - df_loco['y_mean']

    p_raw, _ = pearsonr(df_loco['y_true'], df_loco['y_pred'])
    r2_raw = r2_score(df_loco['y_true'], df_loco['y_pred'])
    p_resid, _ = pearsonr(df_loco['y_true_resid'], df_loco['y_pred_resid'])
    r2_resid = r2_score(df_loco['y_true_resid'], df_loco['y_pred_resid'])
    p_resid_p1, _ = pearsonr(df_loco['y_true_resid'], df_loco['y_pred_p1_resid'])
    r2_resid_p1 = r2_score(df_loco['y_true_resid'], df_loco['y_pred_p1_resid'])
    p_resid_p2 = pearsonr(df_loco['y_true_resid'], df_loco['y_pred_p2_resid'])[0] if 'y_pred_p2_resid' in df_loco.columns else np.nan
    r2_resid_p2 = r2_score(df_loco['y_true_resid'], df_loco['y_pred_p2_resid']) if 'y_pred_p2_resid' in df_loco.columns else np.nan

    per_drug = []
    for drug, g in df_loco.groupby('drug_name'):
        if len(g) >= 10:
            rho_m, _ = spearmanr(g['y_true'], g['y_pred'])
            rho_r, _ = spearmanr(g['y_true_resid'], g['y_pred_resid'])
            rho_r_p1, _ = spearmanr(g['y_true_resid'], g['y_pred_p1_resid'])
            rho_b, _ = spearmanr(g['y_true'], g['y_mean'])
            rec = {'drug': drug, 'n_cells': len(g), 'spearman_model': rho_m, 'spearman_resid': rho_r, 'spearman_resid_p1': rho_r_p1, 'spearman_baseline': rho_b}
            if 'y_pred_p2_resid' in g.columns:
                rho_r_p2, _ = spearmanr(g['y_true_resid'], g['y_pred_p2_resid'])
                rec['spearman_resid_p2'] = rho_r_p2
            per_drug.append(rec)
    df_drug = pd.DataFrame(per_drug)

    median_rho = np.nanmedian(df_drug['spearman_model'])
    median_rho_resid = np.nanmedian(df_drug['spearman_resid'])
    median_rho_resid_p1 = np.nanmedian(df_drug['spearman_resid_p1'])
    median_rho_resid_p2 = np.nanmedian(df_drug['spearman_resid_p2']) if 'spearman_resid_p2' in df_drug.columns else np.nan
    median_rho_base = np.nanmedian(df_drug['spearman_baseline'])
    frac_03 = (df_drug['spearman_model'] > 0.3).mean()
    frac_05 = (df_drug['spearman_model'] > 0.5).mean()

    p_raw_p1, _ = pearsonr(df_loco['y_true'], df_loco['y_pred_p1'])
    r2_raw_p1 = r2_score(df_loco['y_true'], df_loco['y_pred_p1'])
    metrics_row = {'overall_pearson': p_raw, 'overall_r2': r2_raw, 'overall_pearson_p1': p_raw_p1, 'overall_r2_p1': r2_raw_p1, 'resid_pearson': p_resid, 'resid_r2': r2_resid, 'resid_pearson_p1': p_resid_p1, 'resid_r2_p1': r2_resid_p1, 'median_spearman': median_rho, 'median_spearman_resid': median_rho_resid, 'median_spearman_resid_p1': median_rho_resid_p1, 'median_baseline_spearman': median_rho_base, 'frac_rho_gt_03': frac_03, 'frac_rho_gt_05': frac_05}
    metrics_row['resid_pearson_p2'] = p_resid_p2
    metrics_row['resid_r2_p2'] = r2_resid_p2
    metrics_row['median_spearman_resid_p2'] = median_rho_resid_p2
    pd.DataFrame([metrics_row]).to_csv(metrics_path, index=False)
    drug_path = viz_dir / "tnbc_per_drug_spearman.csv"
    df_drug.to_csv(drug_path, index=False)

df_metrics = load_model_metrics()

if df_metrics is None or df_metrics.empty:
    raise RuntimeError(f"df_metrics is None or empty. Metrics file exists: {metrics_path.exists()}, Path: {metrics_path}")

plot_overall_vs_residualized_pearson(df_metrics, output_dir)
plot_overall_vs_residualized_r2(df_metrics, output_dir)

### Residualised Metric Summary

In [ ]:
# Residualized DrEval Comparison Values
# Prints all comparison metrics except baseline/statistical power

if df_metrics is not None and not df_metrics.empty:
    print("=" * 60)
    print("Residualized DrEval Metrics by Model")
    print("=" * 60)
    for _, row in df_metrics.iterrows():
        m = row['model']
        print(f"\n{m}:")
        for col in ['resid_pearson', 'resid_r2', 'median_spearman_resid', 'frac_rho_gt_03', 'frac_rho_gt_05']:
            if col in row.index and pd.notna(row[col]):
                label = {'resid_pearson': 'Residualized Pearson r', 'resid_r2': 'Residualized R²',
                         'median_spearman_resid': 'Median per-drug Spearman ρ (resid)',
                         'frac_rho_gt_03': 'Fraction drugs ρ > 0.3', 'frac_rho_gt_05': 'Fraction drugs ρ > 0.5'}.get(col, col)
                print(f"  {label}: {row[col]:.4f}")

if dreval_p1_vs_p3 is not None:
    print("\n" + "=" * 60)
    print("Phase 1 vs Phase 2 vs Phase 3 GNN (Residualized)")
    print("=" * 60)
    for name, (p1, p2, p3) in dreval_p1_vs_p3.items():
        p2_str = f"{p2:.4f}" if np.isfinite(p2) else "N/A"
        print(f"  {name}: Phase 1 = {p1:.4f}, Phase 2 = {p2_str}, Phase 3 = {p3:.4f}")
print("=" * 60)

In [ ]:
# Classification Head Performance (Phase 3 GNN)
# Binary sensitive vs resistant (per-drug median threshold)
df_cls = df_gnn_pred if df_gnn_pred is not None else None
if df_cls is not None and not df_cls.empty and 'y_true_class' in df_cls.columns and 'y_pred_class' in df_cls.columns:
    yt, yp = df_cls['y_true_class'].values, df_cls['y_pred_class'].values
    tp = int(((yt == 1) & (yp == 1)).sum())
    tn = int(((yt == 0) & (yp == 0)).sum())
    fp = int(((yt == 0) & (yp == 1)).sum())
    fn = int(((yt == 1) & (yp == 0)).sum())
    correct = tp + tn
    wrong = fp + fn
    n = len(yt)
    acc = correct / n if n > 0 else 0
    print("=" * 50)
    print("Classification Head (Phase 3 GNN)")
    print("=" * 50)
    print(f"Correct: {correct}  |  Wrong: {wrong}  |  Total: {n}")
    print(f"Accuracy: {acc:.4f}")
    print(f"TP: {tp}  |  FP: {fp}  |  TN: {tn}  |  FN: {fn}")
    print("=" * 50)
else:
    print("Classification head metrics require tnbc_gnn_loco_predictions.csv with y_true_class and y_pred_class. Run the GNN predictions cell.")

## 9. Residualised Metrics by Phase

Residualised Pearson and R² for each training phase, showing the incremental benefit of Breast and TNBC fine-tuning beyond the drug-mean baseline.

In [ ]:
def plot_residualized_metrics_by_phase(output_dir):
    """Grouped bar: Residualized Pearson and R² for Phase 1, Phase 2, Phase 3."""
    dreval_path = viz_dir / "tnbc_loco_dreval_metrics.csv"
    if not dreval_path.exists():
        dreval_path = output_dir / "tnbc_loco_dreval_metrics.csv"
    if not dreval_path.exists():
        return
    d = pd.read_csv(dreval_path).iloc[0]
    phases = ['Phase 1', 'Phase 2', 'Phase 3']
    resid_pearson = [
        d.get('resid_pearson_p1', np.nan),
        d.get('resid_pearson_p2', np.nan),
        d.get('resid_pearson', np.nan),
    ]
    resid_r2 = [
        d.get('resid_r2_p1', np.nan),
        d.get('resid_r2_p2', np.nan),
        d.get('resid_r2', np.nan),
    ]
    x = np.arange(3)
    w = 0.35
    fig, ax = plt.subplots(figsize=(6, 4), facecolor='white')
    ax.set_facecolor('white')
    ax.yaxis.grid(True, color='#C0C0C0', linestyle='--', linewidth=0.5)
    ax.set_axisbelow(True)
    b1 = ax.bar(x - w/2, resid_pearson, width=w, color='white', edgecolor='black', hatch='/', linewidth=0.8, label='Resid Pearson')
    b2 = ax.bar(x + w/2, resid_r2, width=w, color='white', edgecolor='black', linewidth=0.8, label='Resid R²')
    _label_bars(ax, b1, resid_pearson)
    _label_bars(ax, b2, resid_r2)
    ax.axhline(0, color='black', linewidth=0.5)
    ax.set_xticks(x)
    ax.set_xticklabels(phases)
    ax.set_ylabel('Value')
    ax.set_title('')
    ax.legend(loc=_best_legend_loc(ax), frameon=False)
    vals = [v for v in resid_pearson + resid_r2 if np.isfinite(v)]
    y_min = min(vals) - 0.1 if vals else -0.2
    y_max = max(vals) + 0.1 if vals else 0.2
    ax.set_ylim(y_min, max(y_max, 0.05))
    plt.tight_layout()
    fig.savefig(output_dir / 'tnbc_residualized_by_phase.pdf', facecolor='white', bbox_inches='tight')
    fig.savefig(output_dir / 'tnbc_residualized_by_phase.png', facecolor='white', bbox_inches='tight')
    plt.show()
    plt.close()

plot_residualized_metrics_by_phase(output_dir)

## 10. Predicted vs Actual Scatter Plots

Predicted vs true LN_IC50 for Phase 3 GNN, Random Forest, and ElasticNet. Coloured by cell line or drug class to visualise systematic biases.

In [ ]:
def plot_predicted_vs_actual(df_pred, output_dir, color_by='cell_line', drug_to_class=None, suffix=''):
    """Scatter: predicted LN_IC50 vs true LN_IC50. Monochrome, matches other viz style.
    """
    if df_pred is None or df_pred.empty:
        return
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.set_facecolor('white')
    ax.yaxis.grid(True, color='#C0C0C0', linestyle='--', linewidth=0.5)
    ax.xaxis.grid(True, color='#C0C0C0', linestyle='--', linewidth=0.5)
    ax.set_axisbelow(True)
    x = df_pred['y_true'].values
    y = df_pred['y_pred'].values

    max_legend_groups = 8
    markers = ['o', 's', '^', 'D', 'v', '<', '>', 'p']
    used_grouped = False
    if color_by == 'drug_class' and drug_to_class:
        df_pred = df_pred.copy()
        df_pred['drug_class'] = df_pred['drug_name'].map(
            lambda d: drug_to_class.get(d, 'Unknown') if isinstance(d, str) else 'Unknown'
        )
        groups = list(df_pred.groupby('drug_class'))
        n_groups = len(groups)
        if n_groups <= max_legend_groups:
            used_grouped = True
            for i, (name, g) in enumerate(groups):
                ax.scatter(g['y_true'], g['y_pred'], alpha=0.6, s=20, label=name,
                          c='black', marker=markers[i % len(markers)], edgecolors='none')
            ax.legend(loc=_best_legend_loc(ax), frameon=False)
        else:
            ax.scatter(x, y, alpha=0.5, s=20, c='black', edgecolors='none')
    else:
        ax.scatter(x, y, alpha=0.5, s=20, c='black', edgecolors='none')

    lim_min = min(x.min(), y.min()) - 0.5
    lim_max = max(x.max(), y.max()) + 0.5
    ax.plot([lim_min, lim_max], [lim_min, lim_max], 'k--', linewidth=1, alpha=0.7)
    ax.set_xlabel('True LN_IC50')
    ax.set_ylabel('Predicted LN_IC50')
    subtitle = f'Coloured by {color_by.replace("_", " ")}' if used_grouped else ''
    ax.set_title('')
    ax.set_xlim(lim_min, lim_max)
    ax.set_ylim(lim_min, lim_max)
    ax.set_aspect('equal')
    r2_val = r2_score(df_pred['y_true'], df_pred['y_pred']) if 'y_true' in df_pred.columns and 'y_pred' in df_pred.columns else np.corrcoef(x, y)[0, 1] ** 2
    r2_label = f'R² = {r2_val:.3f}'
    ax.text(0.05, 0.95, r2_label, transform=ax.transAxes, fontsize=10, va='top')
    plt.tight_layout()
    base = "tnbc_predicted_vs_actual" + (f"_{suffix}" if suffix else "")
    fig.savefig(output_dir / f"{base}.pdf", facecolor='white', bbox_inches='tight')
    fig.savefig(output_dir / f"{base}.png", facecolor='white', bbox_inches='tight')
    plt.show()
    plt.close()

### Multi-Model Scatter and Residual Plots

In [ ]:
# Marker per model for predicted vs actual (black and white)
PRED_VS_ACTUAL_STYLE = {
    'Phase 3 GNN': {'marker': 'o', 's': 32},
    'Random Forest': {'marker': 's', 's': 28},
    'ElasticNet': {'marker': '^', 's': 32},
}

def plot_predicted_vs_actual_all_models(df_gnn, df_rf, df_en, output_dir):
    """Three separate plots + one combined overlay: predicted vs actual scatter for GNN, RF, ElasticNet."""
    preds = [("Phase 3 GNN", df_gnn, "phase3_gnn"), ("Random Forest", df_rf, "random_forest"), ("ElasticNet", df_en, "elasticnet")]
    preds = [(n, d, slug) for n, d, slug in preds if d is not None and not d.empty and 'y_true' in d.columns and 'y_pred' in d.columns]
    if not preds:
        return
    all_lims = []
    for name, df, slug in preds:
        x, y = df['y_true'].values, df['y_pred'].values
        lim_min = min(x.min(), y.min()) - 0.5
        lim_max = max(x.max(), y.max()) + 0.5
        all_lims.extend([lim_min, lim_max])
    lim_lo, lim_hi = min(all_lims), max(all_lims)
    for name, df, slug in preds:
        fig, ax = plt.subplots(figsize=(6, 6))
        x, y = df['y_true'].values, df['y_pred'].values
        ax.set_facecolor('white')
        ax.yaxis.grid(True, color='#C0C0C0', linestyle='--', linewidth=0.5)
        ax.xaxis.grid(True, color='#C0C0C0', linestyle='--', linewidth=0.5)
        ax.set_axisbelow(True)
        style = PRED_VS_ACTUAL_STYLE.get(name, {'marker': 'o', 's': 16})
        ax.scatter(x, y, alpha=0.6, s=style['s'], c='black', marker=style['marker'], edgecolors='none')
        ax.plot([lim_lo, lim_hi], [lim_lo, lim_hi], 'k--', linewidth=1, alpha=0.7)
        ax.set_xlabel('True LN_IC50')
        ax.set_ylabel('Predicted LN_IC50')
        ax.set_xlim(lim_lo, lim_hi)
        ax.set_ylim(lim_lo, lim_hi)
        ax.set_aspect('equal')
        r2_val = r2_score(df['y_true'], df['y_pred']) if 'y_true' in df.columns and 'y_pred' in df.columns else np.corrcoef(x, y)[0, 1] ** 2
        ax.text(0.05, 0.95, f'R² = {r2_val:.3f}', transform=ax.transAxes, fontsize=10, va='top')
        plt.tight_layout()
        base = f'tnbc_predicted_vs_actual_{slug}'
        fig.savefig(output_dir / f'{base}.pdf', facecolor='white', bbox_inches='tight')
        fig.savefig(output_dir / f'{base}.png', facecolor='white', bbox_inches='tight')
        plt.show()
        plt.close()

    # Combined overlay: all models on one plot with distinct markers and colors
    fig, ax = plt.subplots(figsize=(6.5, 6))
    ax.set_facecolor('white')
    ax.yaxis.grid(True, color='#C0C0C0', linestyle='--', linewidth=0.5)
    ax.xaxis.grid(True, color='#C0C0C0', linestyle='--', linewidth=0.5)
    ax.set_axisbelow(True)
    for name, df, _ in preds:
        x, y = df['y_true'].values, df['y_pred'].values
        style = PRED_VS_ACTUAL_STYLE.get(name, {'marker': 'o', 's': 28})
        r2_val = r2_score(df['y_true'], df['y_pred'])
        ax.scatter(x, y, alpha=0.55, s=style['s'], c='black', marker=style['marker'],
                   edgecolors='none', label=f'{name} (R²={r2_val:.3f})')
    ax.plot([lim_lo, lim_hi], [lim_lo, lim_hi], 'k--', linewidth=1, alpha=0.7)
    ax.set_xlabel('True LN_IC50')
    ax.set_ylabel('Predicted LN_IC50')
    ax.set_xlim(lim_lo, lim_hi)
    ax.set_ylim(lim_lo, lim_hi)
    ax.set_aspect('equal')
    ax.legend(loc=_best_legend_loc(ax), frameon=False)
    plt.tight_layout()
    fig.savefig(output_dir / 'tnbc_predicted_vs_actual_combined.pdf', facecolor='white', bbox_inches='tight')
    fig.savefig(output_dir / 'tnbc_predicted_vs_actual_combined.png', facecolor='white', bbox_inches='tight')
    plt.show()
    plt.close()

def plot_residual_distribution_comparison(df_gnn, df_rf, df_en, output_dir):
    """Boxplot: residual (y_pred - y_true) distribution per model. Monochrome style."""
    preds = [("Phase 3 GNN", df_gnn), ("Random Forest", df_rf), ("ElasticNet", df_en)]
    preds = [(n, d) for n, d in preds if d is not None and not d.empty and 'y_true' in d.columns and 'y_pred' in d.columns]
    if not preds:
        return
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.set_facecolor('white')
    ax.yaxis.grid(True, color='#C0C0C0', linestyle='--', linewidth=0.5)
    ax.set_axisbelow(True)
    hatches = {'Phase 3 GNN': '/', 'Random Forest': '', 'ElasticNet': 'xx'}
    res_data = [pred[1]['y_pred'].values - pred[1]['y_true'].values for pred in preds]
    labels = [pred[0] for pred in preds]
    bp = ax.boxplot(res_data, labels=labels, patch_artist=True, widths=0.6)
    for i, (patch, lab) in enumerate(zip(bp['boxes'], labels)):
        patch.set_facecolor('white')
        patch.set_edgecolor('black')
        patch.set_hatch(hatches.get(lab, ''))
    for el in bp['medians'] + bp['whiskers'] + bp['caps']:
        el.set_color('black')
    ax.axhline(0, color='black', linewidth=0.5, linestyle='--')
    ax.set_ylabel('Residual (Predicted − True LN_IC50)')
    ax.set_xlabel('Model')
    ax.set_title('')
    plt.tight_layout()
    fig.savefig(output_dir / 'tnbc_residual_distribution_comparison.pdf', facecolor='white', bbox_inches='tight')
    fig.savefig(output_dir / 'tnbc_residual_distribution_comparison.png', facecolor='white', bbox_inches='tight')
    plt.show()
    plt.close()

def plot_prediction_residual_distribution(df_pred, output_dir):
    """Histogram of prediction residuals (y_pred - y_true). Monochrome, matches other viz style.
    """
    if df_pred is None or df_pred.empty:
        return
    residuals = df_pred['y_pred'].values - df_pred['y_true'].values
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.set_facecolor('white')
    ax.yaxis.grid(True, color='#C0C0C0', linestyle='--', linewidth=0.5)
    ax.xaxis.grid(True, color='#C0C0C0', linestyle='--', linewidth=0.5)
    ax.set_axisbelow(True)
    ax.hist(residuals, bins=30, color='white', edgecolor='black', linewidth=0.8)
    ax.axvline(0, color='black', linestyle='--', linewidth=1, alpha=0.7)
    ax.set_xlabel('Residual (Predicted − True LN_IC50)')
    ax.set_ylabel('Count')
    ax.set_title('')
    plt.tight_layout()
    fig.savefig(output_dir / 'tnbc_predicted_vs_actual_residual_dist.pdf', facecolor='white', bbox_inches='tight')
    fig.savefig(output_dir / 'tnbc_predicted_vs_actual_residual_dist.png', facecolor='white', bbox_inches='tight')
    plt.show()
    plt.close()

### Execute Scatter Plots

In [ ]:
if df_gnn_pred is not None and not df_gnn_pred.empty:
    if drug_to_class:
        n_drug_classes = df_gnn_pred['drug_name'].map(
            lambda d: drug_to_class.get(d, 'Unknown') if isinstance(d, str) else 'Unknown'
        ).nunique()
        if n_drug_classes <= 8:
            plot_predicted_vs_actual(df_gnn_pred, output_dir, color_by='drug_class', drug_to_class=drug_to_class, suffix='drug_class')
        else:
            plot_predicted_vs_actual(df_gnn_pred, output_dir, color_by='cell_line', suffix='')
    else:
        plot_predicted_vs_actual(df_gnn_pred, output_dir, color_by='cell_line', suffix='')
else:
    print(f"Warning: df_gnn_pred is {'None' if df_gnn_pred is None else 'empty'}, skipping predicted vs actual plots")
if any(d is not None and not d.empty for d in [df_gnn_pred, df_rf_pred, df_en_pred]):
    plot_predicted_vs_actual_all_models(df_gnn_pred, df_rf_pred, df_en_pred, output_dir)
    plot_residual_distribution_comparison(df_gnn_pred, df_rf_pred, df_en_pred, output_dir)

## 11. Statistical Tests

Friedman test across GNN, Random Forest, and ElasticNet fold-level MSEs, followed by pairwise Wilcoxon signed-rank tests with Bonferroni correction.

In [ ]:
# Load fold-level MSEs
# 1) GNN from fold_results in memory
if 'fold_results' in globals() and fold_results:
    gnn_mses = []
    for r in fold_results:
        if 'phase3' in r and 'mse' in r['phase3']:
            gnn_mses.append(r['phase3']['mse'])
        elif 'phase3' in r and 'rmse' in r['phase3']:
            # Compute MSE from RMSE: MSE = RMSE²
            gnn_mses.append(r['phase3']['rmse'] ** 2)
        else:
            raise ValueError("fold_results missing phase3 mse or rmse")
    n_folds = len(gnn_mses)
else:
    raise RuntimeError("fold_results not found in memory")

# 2) RF and EN from CSV files or prediction files
rf_mses = None
en_mses = None

# Try CSV files first (check both naming conventions)
# CSV files typically have one row with overall metrics, but check for per-fold data
for rf_path in [output_dir / "rf_loco_metrics.csv", output_dir / "tnbc_rf_loco_metrics.csv"]:
    if rf_path.exists():
        df_rf_metrics = pd.read_csv(rf_path)
        if 'mse' in df_rf_metrics.columns and len(df_rf_metrics) == n_folds:
            rf_mses = df_rf_metrics['mse'].values
            break
        elif 'fold_mse' in df_rf_metrics.columns and len(df_rf_metrics) == n_folds:
            rf_mses = df_rf_metrics['fold_mse'].values
            break

for en_path in [output_dir / "en_loco_metrics.csv", output_dir / "tnbc_elasticnet_loco_metrics.csv"]:
    if en_path.exists():
        df_en_metrics = pd.read_csv(en_path)
        if 'mse' in df_en_metrics.columns and len(df_en_metrics) == n_folds:
            en_mses = df_en_metrics['mse'].values
            break
        elif 'fold_mse' in df_en_metrics.columns and len(df_en_metrics) == n_folds:
            en_mses = df_en_metrics['fold_mse'].values
            break

# If CSV doesn't have fold-level MSE, compute from prediction files
if rf_mses is None:
    for rf_pred_path in [output_dir / "rf_loco_predictions.csv", output_dir / "tnbc_rf_loco_predictions.csv"]:
        if rf_pred_path.exists():
            df_rf_pred = pd.read_csv(rf_pred_path)
            if 'fold_id' in df_rf_pred.columns and 'y_true' in df_rf_pred.columns and 'y_pred' in df_rf_pred.columns:
                # Compute MSE per fold, sorted by fold_id to match fold_results order
                rf_fold_mses = df_rf_pred.groupby('fold_id').apply(
                    lambda g: mean_squared_error(g['y_true'], g['y_pred'])
                ).sort_index()
                if len(rf_fold_mses) == n_folds:
                    rf_mses = rf_fold_mses.values
                    break

if en_mses is None:
    for en_pred_path in [output_dir / "en_loco_predictions.csv", output_dir / "tnbc_elasticnet_loco_predictions.csv"]:
        if en_pred_path.exists():
            df_en_pred = pd.read_csv(en_pred_path)
            if 'fold_id' in df_en_pred.columns and 'y_true' in df_en_pred.columns and 'y_pred' in df_en_pred.columns:
                # Compute MSE per fold, sorted by fold_id to match fold_results order
                en_fold_mses = df_en_pred.groupby('fold_id').apply(
                    lambda g: mean_squared_error(g['y_true'], g['y_pred'])
                ).sort_index()
                if len(en_fold_mses) == n_folds:
                    en_mses = en_fold_mses.values
                    break

# Fallback to pickle file (unlikely to have RF/EN results, but check)
if rf_mses is None or en_mses is None:
        loco_pkl_path = output_dir / "loco_cv_results_cell_line.pkl"
    if loco_pkl_path.exists():
        with open(loco_pkl_path, 'rb') as f:
            loco_data = pickle.load(f)
        # Pickle typically only has GNN fold_results, not RF/EN

# Verify we have all three arrays
if rf_mses is None:
    raise RuntimeError("Could not load RF fold-level MSEs from CSV, prediction files, or pickle")
if en_mses is None:
    raise RuntimeError("Could not load ElasticNet fold-level MSEs from CSV, prediction files, or pickle")

# Ensure all arrays have the same length (21 folds)
if len(rf_mses) != n_folds or len(en_mses) != n_folds:
    raise ValueError(f"Mismatch in fold counts: GNN={n_folds}, RF={len(rf_mses)}, EN={len(en_mses)}")

# Run Friedman test
stat, p_friedman = friedmanchisquare(gnn_mses, rf_mses, en_mses)

# Run pairwise Wilcoxon tests with Bonferroni correction if Friedman is significant
pairs = [
    ("GNN vs RF", gnn_mses, rf_mses),
    ("GNN vs EN", gnn_mses, en_mses),
    ("RF vs EN", rf_mses, en_mses)
]
pairwise_results = []
if p_friedman < 0.05:
    for name, x, y in pairs:
        stat_w, p_w = wilcoxon(x, y, alternative='two-sided')
        pairwise_results.append((name, p_w))

    # Bonferroni correction
    n_comparisons = len(pairwise_results)
    pairwise_results = [(name, p * n_comparisons) for name, p in pairwise_results]

# Format significance stars
def format_pvalue(p):
    if p < 0.001:
        return f"{p:.4f} ***"
    elif p < 0.01:
        return f"{p:.4f} **"
    elif p < 0.05:
        return f"{p:.4f} *"
    else:
        return f"{p:.4f} ns"

# Print summary table
print("=" * 60)
print("Friedman Test + Post-hoc Significance Analysis")
print("=" * 60)
print(f"\nFriedman test p-value: {format_pvalue(p_friedman)}")
if p_friedman < 0.05:
    print(f"\nPairwise Wilcoxon tests (Bonferroni corrected):")
    print(f"{'Comparison':<15} {'p-value':<20}")
    print("-" * 35)
    for name, p_corrected in pairwise_results:
        print(f"{name:<15} {format_pvalue(p_corrected):<20}")
else:
    print("\nFriedman test not significant (p >= 0.05), skipping post-hoc tests.")
print("=" * 60)